[Markdown Cell]

# Phase 4 / 01 A Full FAST

Attach (1) completed Phase 4 notebook 00 output, (2) your complete **Phase 1 Model A foundation output**, (3) its exact tokenizer.json and metadata.json if absent from that output. A previous Phase 2/3 task can supply an unchanged tokenizer snapshot only; its NER/RR weights cannot initialize CJPE. Select **T4 x2**, Internet ON, and Run All in a fresh kernel.

The original own foundation defaults to 4 completed LM epochs. This notebook verifies actual checkpoint bytes, sources and tokenizer. A .ipynb attachment or observations.zip does not contain the required tensors. Do not attach multiple conflicting versions; explicit paths below resolve ambiguity.

**Optimized T4 x2 engine:** every original training row, content token/chunk, trainable parameter and global batch is preserved. Encoder packs default to 8 chunks (use 4 if memory is tight). Overlap one CPU batch with GPU work; keep FP16/NCCL DDP and full 500-step/epoch recovery. Inference uses both GPUs.

**Continue a previously started original run:** set UPGRADE_FROM_RUN_DIR to its complete saved Phase 4 root; attach its full weights/receipts/code/tokenizer plus prepared data. Keep the original learning choices, chunk length, head and global case batch. No Phase 1 checkpoint is needed when full task state is supplied. This creates a new audited engine identity and preserves progress/optimizer; numerical trajectories can differ after the transition. For later resumes of this FAST run use RESUME_RUN_DIR, leaving UPGRADE_FROM_RUN_DIR=None.


**V2 fix:** GRU sequence lengths stay on CPU across DDP's CUDA argument transfer. The new experiment ID ends in FAST_V2 so the previous engine's saved source fingerprint is not overwritten. The reported first-step failure performed zero optimizer updates. For that failed run, leave both recovery paths None and reuse the same inputs; attaching its complete saved cache avoids tokenizing again. For a previously progressed FAST v1 run, use UPGRADE_FROM_RUN_DIR with its complete unchanged saved output.


[Markdown Cell]

## 1. Environment

In [1]:
# [Code Cell]
import importlib.metadata as package_metadata
import json
import os
from pathlib import Path
import subprocess
import sys

requirements = {'tokenizers': 'tokenizers==0.23.2', 'numpy': 'numpy==2.1.3',
                'scikit-learn': 'scikit-learn==1.7.2', 'tqdm': 'tqdm>=4.66,<5',
                'huggingface-hub': 'huggingface_hub>=0.30,<2', 'ijson': 'ijson>=3.3,<4'}
install = []
for package, requirement in requirements.items():
    try:
        version = package_metadata.version(package)
        if '==' in requirement and version != requirement.split('==')[1]:
            install.append(requirement)
    except package_metadata.PackageNotFoundError:
        install.append(requirement)
if install:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *install])
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
print('Dependencies ready; Kaggle PyTorch is preserved. Use a fresh kernel before Run All.')


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 68.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.4/9.4 MB 108.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 150.2/150.2 kB 7.8 MB/s eta 0:00:00
Dependencies ready; Kaggle PyTorch is preserved. Use a fresh kernel before Run All.


[Markdown Cell]

## 2. Input paths, time and exact continuation

In [2]:
# [Code Cell]
EXPERIMENT_ID='P4_01_A_Full_FAST_V2'
FAMILY='a'
METHOD='full'
OUTPUT_ROOT='/kaggle/working/cs781_phase4/'+EXPERIMENT_ID
PREPARED_DIR=None
PARENT_RUN_DIR=None
TOKENIZER_PATH=None
METADATA_PATH=None
RESUME_RUN_DIR=None
TEACHER_RUN_DIR=None
EXPECTED_PARENT_SHA256='0746fa1981f804a246b65d82f1d13e35c3a6bdf554bfff69df26a35dd1d5d131'
MAX_EPOCHS=4
TARGET_TOTAL_EPOCHS=None
TRAINING_MINUTES=300
CHUNK_TOKENS=256
CHUNK_MICRO_BATCH=8
BACKBONE_LR=2e-05
HEAD_LR=2e-4
MODE='train'
Path(OUTPUT_ROOT).mkdir(parents=True,exist_ok=True)

UPGRADE_FROM_RUN_DIR=None
PREFETCH_BATCHES=True
LOG_EVERY_STEPS=10
INFERENCE_CASES_PER_BATCH=4
INFERENCE_CHUNK_BUDGET=64


[Markdown Cell]

`None` paths enable discovery. Explicit examples: PREPARED_DIR is the folder containing prepared_manifest.json (or phase4_prepared.zip); PARENT_RUN_DIR contains train_config.json, code/model_a (or b/c/d) and checkpoints; RESUME_RUN_DIR is a previous **same experiment CJPE** root containing job.json and real checkpoints.

To continue a timed stop, attach the full saved output, keep learning choices unchanged and Run All. To train one more whole epoch after completion, set TARGET_TOTAL_EPOCHS to completed epoch count + 1. The original LR schedule is preserved. Change the experiment ID for different learning/head/chunk settings. Leave MAX_EPOCHS unchanged during recovery. A lower target may cap a fresh run without changing its original scheduler horizon; training never drops cases to meet a time limit.

For an additional verified own Phase 1 continuation of the same architecture/data contract, set EXPECTED_PARENT_SHA256=None and the explicit parent path. A newly redesigned/full-corpus foundation needs a separately audited contract; do not bypass source/tokenizer checks.

In [3]:
# [Code Cell]
module_parts={}
def append_module(name, source):
    module_parts.setdefault(name, []).append(source)


[Markdown Cell]

### __init__.py

Typed settings, strict serialization and verified own-foundation loading.

[Markdown Cell]

Importable source part 1

In [4]:
# [Code Cell]
append_module('__init__.py', r'''VERSION = "cs781-cjpe-v1"
''')


[Markdown Cell]

### adaptation.py

Five distinct methods: full A; identity-initialized LoRMA+ $W=(I+\alpha BA/r)W_0$ on A; actual encoder biases only on B (BitFit); nonlinear residual bottlenecks on frozen C; final two D layers plus final norm. KD trains the active B encoder against a larger learned own CJPE teacher. Every method also learns the same case head.

[Markdown Cell]

Importable source part 1

In [5]:
# [Code Cell]
append_module('adaptation.py', r'''# CELL: Five genuinely distinct active-foundation adaptation mechanisms
import math
import torch
from torch import nn
from torch.nn import functional as F


class LoRMA(nn.Module):
    """Frozen W0 plus multiplicative low-rank update: (I + alpha/r BA) W0 x + b."""
    def __init__(self, base: nn.Linear, rank: int, alpha: float):
        super().__init__()
        self.base = base
        self.base.requires_grad_(False)
        self.a = nn.Parameter(torch.empty(rank, base.out_features))
        self.b = nn.Parameter(torch.zeros(base.out_features, rank))
        self.scale = alpha / rank
        nn.init.kaiming_uniform_(self.a, a=math.sqrt(5))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        hidden = F.linear(x, self.base.weight, None)
        output = hidden + self.scale * F.linear(F.linear(hidden, self.a), self.b)
        return output if self.base.bias is None else output + self.base.bias


class ResidualAdapter(nn.Module):
    """Frozen projection y plus a learned GELU bottleneck residual, initially zero."""
    def __init__(self, base: nn.Linear, width: int):
        super().__init__()
        self.base = base
        self.base.requires_grad_(False)
        self.down = nn.Linear(base.out_features, width)
        self.up = nn.Linear(width, base.out_features)
        nn.init.zeros_(self.up.weight)
        nn.init.zeros_(self.up.bias)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        hidden = self.base(x)
        return hidden + self.up(F.gelu(self.down(hidden)))


def adapt(backbone: nn.Module, cfg) -> list[str]:
    """Adapt active feature layers only; B's unused autoregressive decoder is excluded."""
    backbone.requires_grad_(False)
    if cfg.method in ('full', 'kd'):
        backbone.requires_grad_(True)
        if cfg.family == 'b':
            for module in (backbone.decoder, backbone.decoder_norm, backbone.target_positions):
                module.requires_grad_(False)
''')


[Markdown Cell]

Importable source part 2

In [6]:
# [Code Cell]
append_module('adaptation.py', r'''        return [name for name, p in backbone.named_parameters() if p.requires_grad]
    if cfg.method == 'bitfit':
        names = []
        for name, parameter in backbone.named_parameters():
            if name.endswith('.bias') and name.startswith(('encoder.', 'encoder_norm.')):
                parameter.requires_grad_(True)
                names.append(name)
        if not names:
            raise ValueError('BitFit found no real active encoder biases; refusing head-only training')
        return names
    if cfg.method == 'partial':
        if cfg.partial_layers > len(backbone.layers):
            raise ValueError('partial_layers exceeds the actual D depth')
        for layer in backbone.layers[-cfg.partial_layers:]:
            layer.requires_grad_(True)
        backbone.final_norm.requires_grad_(True)
        return [name for name, p in backbone.named_parameters() if p.requires_grad]
    targets = list(backbone.named_modules())
    names = []
    for name, module in targets:
        replacement = None
        if cfg.method == 'lorma' and name.startswith('layers.') and isinstance(module, nn.Linear):
            replacement = LoRMA(module, cfg.rank, cfg.alpha)
        if (cfg.method == 'adapters' and name.startswith('stages.') and isinstance(module, nn.Linear)
                and name.endswith(('attention.out', 'ffn.down'))):
            replacement = ResidualAdapter(module, cfg.adapter_width)
        if replacement is not None:
            parent, _, child = name.rpartition('.')
            setattr(backbone.get_submodule(parent), child, replacement)
            names.append(name)
    if not names:
        raise ValueError('No intended backbone adaptation modules were found')
    return names
''')


[Markdown Cell]

### analysis.py

Dev-only frozen-weight ablations compare whole-document attention, whole-document mean and first/last sections, with coverage and length/error diagnostics. This isolates inference representation sensitivity; separately trained representation trials would still be needed for a causal training comparison.

[Markdown Cell]

Importable source part 1

In [7]:
# [Code Cell]
append_module('analysis.py', r'''# CELL: Frozen-weight dev-only representation ablations, not extra tuning methods
import argparse
from dataclasses import replace
from pathlib import Path
import numpy as np
import torch
from .data import CaseDataset, collate
from .io_utils import atomic_json, read_json
from .metrics import case_diagnostics, probabilities
from .model import forward
from .predict import load_task, logits_partition, ordered_logits
from .select import completed_outputs


def analyze(input_root, prepared, output, task_root=None):
    output = Path(output)
    output.mkdir(parents=True, exist_ok=True)
    candidates, _, _ = completed_outputs(input_root, prepared, [task_root] if task_root else None)
    chosen = max(candidates, key=lambda c: c['receipt']['raw_dev_macro_f1'])
    model, job, cfg, _ = load_task(chosen['root'], Path(prepared), torch.device('cuda', 0), output / 'inference_cache')
    rows = job['cache']['cases']['dev']
    results = {}
    for name, representation, pooling in (('whole_document_learned_attention', 'all', cfg.pooling),
                                           ('whole_document_mean', 'all', 'mean'),
                                           ('first_last_sections', 'first_last', cfg.pooling)):
        altered = replace(cfg, representation=representation, pooling=pooling)
        model.cfg = altered
        logits = ordered_logits(logits_partition(model, job, altered, 'dev', torch.device('cuda', 0)), rows)
        data = CaseDataset(job['cache'], 'dev', altered)
        kept = total = 0
        for index in range(len(data)):
            value = data[index]
            kept += value['kept_tokens']
            total += value['total_tokens']
        data.close()
        results[name] = {**case_diagnostics([r['gold'] for r in rows], probabilities(logits) >= .5, rows),
                         'content_tokens_kept_fraction': kept / max(1, total)}
    report = {'status': 'analysis_complete', 'task': chosen['receipt']['experiment_id'], 'results': results,
              'scope': 'Inference ablations of identical frozen weights on dev only. This does not establish which representation would win after separate training; no new method is claimed and test predictions are not changed.',
              'test_labels_used': False}
    atomic_json(output / 'representation_analysis.json', report)
    return report


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument('--plan', required=True)
    args = parser.parse_args()
''')


[Markdown Cell]

Importable source part 2

In [8]:
# [Code Cell]
append_module('analysis.py', r'''    print(analyze(**read_json(args.plan)), flush=True)


if __name__ == '__main__':
    main()
''')


[Markdown Cell]

### config.py

Typed settings, strict serialization and verified own-foundation loading.

[Markdown Cell]

Importable source part 1

In [9]:
# [Code Cell]
append_module('config.py', r'''# CELL: Explicit immutable learning choices and portable input paths
from dataclasses import asdict, dataclass


@dataclass(frozen=True)
class CJPEConfig:
    """Whole-case classification; one weighted target per original case."""
    experiment_id: str = 'P4_01_A_Full'
    family: str = 'a'
    method: str = 'full'
    run_dir: str = '/kaggle/working/cs781_phase4/P4_01_A_Full'
    input_root: str = '/kaggle/input'
    prepared_dir: str | None = None
    parent_run_dir: str | None = None
    tokenizer_path: str | None = None
    metadata_path: str | None = None
    resume_run_dir: str | None = None
    upgrade_run_dir: str | None = None
    teacher_run_dir: str | None = None
    expected_parent_sha256: str | None = None
    parent_checkpoint: str = 'best'
    max_epochs: int = 4
    minimum_epochs: int = 2
    patience: int = 2
    target_total_epochs: int | None = None
    seed: int = 781
    world_size: int = 2
    micro_batch_size: int = 1
    accumulation_steps: int = 4
    num_workers: int = 0
    max_run_seconds: int = 180 * 60
    min_free_bytes: int = 1 << 30
    checkpoint_every_steps: int = 500
    max_consecutive_overflows: int = 8
    chunk_tokens: int = 256
    chunk_micro_batch: int = 8
    prefetch_batches: bool = True
    log_every_steps: int = 10
    inference_cases_per_batch: int = 4
    inference_chunk_budget: int = 64
    frozen_prefix_cache_tokens: int = 131072
    head_width: int = 128
    dropout: float = 0.2
    pooling: str = 'attention'
    representation: str = 'all'
    section_chunks: int = 4
    recurrent_chunks: bool = True
    checkpoint_encoder: bool = True
''')


[Markdown Cell]

Importable source part 2

In [10]:
# [Code Cell]
append_module('config.py', r'''    backbone_lr: float = 2e-5
    head_lr: float = 2e-4
    weight_decay: float = 0.01
    warmup_fraction: float = 0.1
    min_lr_ratio: float = 0.1
    max_grad_norm: float = 1.0
    class_weights: str = 'auto'
    rank: int = 16
    alpha: float = 16.0
    adapter_width: int = 32
    partial_layers: int = 2
    distillation_temperature: float = 2.0
    distillation_weight: float = 0.25
    calibrate_threshold: bool = True
    calibration_margin: float = 0.002

    def validate(self) -> None:
        matrix = {'full': 'a', 'lorma': 'a', 'bitfit': 'b', 'adapters': 'c', 'partial': 'd', 'kd': 'b'}
        if matrix.get(self.method) != self.family:
            raise ValueError('Use the five distinct Phase 4 methods across A/B/C/D plus B distillation')
        if not self.experiment_id or any(c in self.experiment_id for c in '/\\'):
            raise ValueError('Use a simple experiment ID without path separators')
        if self.world_size != 2 or min(self.micro_batch_size, self.accumulation_steps) < 1:
            raise ValueError('Use two GPUs and positive accumulation/microbatch sizes')
        if self.num_workers != 0 or self.checkpoint_every_steps != 500:
            raise ValueError('Exact deterministic recovery uses zero loader workers and 500-update checkpoints')
        if min(self.max_epochs, self.minimum_epochs, self.patience) < 1:
            raise ValueError('Epoch counts and patience must be positive')
        if self.target_total_epochs is not None and self.target_total_epochs < 1:
            raise ValueError('Target total epochs must be positive')
        if not 16 <= self.chunk_tokens <= 512 or not 1 <= self.chunk_micro_batch <= 16:
            raise ValueError('Invalid chunk token/microbatch dimensions')
        if not 32 <= self.head_width <= 512 or self.head_width % 2 or not 0 <= self.dropout < 1:
            raise ValueError('Use an even head width and valid dropout')
        if self.pooling not in ('mean', 'attention') or self.representation not in ('all', 'first_last'):
            raise ValueError('Unknown pooling/representation policy')
        if self.section_chunks < 1 or self.partial_layers < 1:
            raise ValueError('Section/chosen-layer counts must be positive')
        if self.class_weights not in ('none', 'sqrt', 'auto'):
            raise ValueError('Class weights must be none/sqrt/auto and use training labels only')
        if not 0 < self.backbone_lr <= 1e-3 or not 0 < self.head_lr <= 1e-2:
            raise ValueError('Invalid learning rates')
        if not 0 < self.distillation_weight < 1 or not 1 <= self.distillation_temperature <= 10:
            raise ValueError('Invalid distillation settings')
        if not 1 <= self.rank <= 128 or not 0 < self.alpha <= 256 or not 4 <= self.adapter_width <= 256:
            raise ValueError('Invalid low-rank/adapter dimensions')
        if self.max_run_seconds < 1 or not 0 <= self.warmup_fraction < 1 or not 0 < self.min_lr_ratio <= 1:
            raise ValueError('Invalid time/scheduler settings')
''')


[Markdown Cell]

Importable source part 3

In [11]:
# [Code Cell]
append_module('config.py', r'''        if self.parent_checkpoint not in ('best', 'latest'):
            raise ValueError('Unknown parent checkpoint policy')
        if self.log_every_steps < 1 or self.inference_cases_per_batch < 1 or self.inference_chunk_budget < 1:
            raise ValueError('Invalid performance reporting/inference budget')
        if self.frozen_prefix_cache_tokens < 1:
            raise ValueError('Frozen-prefix cache token budget must be positive')

    def choices(self) -> dict:
        """Learning identity excludes transport paths and explicit duration requests."""
        value = asdict(self)
        for name in ('run_dir', 'input_root', 'prepared_dir', 'parent_run_dir', 'tokenizer_path', 'metadata_path',
                     'resume_run_dir', 'upgrade_run_dir', 'teacher_run_dir', 'expected_parent_sha256', 'parent_checkpoint',
                     'max_epochs', 'target_total_epochs', 'max_run_seconds', 'min_free_bytes'):
            value.pop(name)
        return value
''')


[Markdown Cell]

### data.py

Each original case owns all its chunks. No document/tail cap is used for the default whole-document baseline. Disk stores uint16 tokens; only one case is encoded at a time. Length-bucketed, exhaustive distributed batches use masked dummy tail cases. One label supervises the whole case, not every chunk separately.

[Markdown Cell]

Importable source part 1

In [12]:
# [Code Cell]
append_module('data.py', r'''# CELL: Lossless uint16 case caches, no hidden document/chunk cap
from array import array
import math
from pathlib import Path
import re
import sys
import unicodedata
import numpy as np
import torch
from tokenizers import Tokenizer
from tqdm.auto import tqdm
from .io_utils import atomic_json, ensure_space, identity, read_json, records, sha256_file


def verify_cache(root: Path, expected: str) -> dict:
    value = read_json(root / 'case_cache_manifest.json')
    if value.get('version') != 'cjpe-cache-v1' or value['identity'] != expected:
        raise ValueError('Wrong case-cache identity')
    if identity(value['choices']) != expected or identity({k:v for k,v in value.items() if k != 'manifest_sha256'}) != value.get('manifest_sha256'):
        raise ValueError('Changed case-cache metadata manifest')
    for name, receipt in value['artifacts'].items():
        if Path(name).name != name or sha256_file(root / name) != receipt['sha256'] or (root / name).stat().st_size != receipt['bytes']:
            raise ValueError('Changed/missing case cache: ' + name)
    return {**value, 'directory': str(root)}


def cache_data(prepared: Path, tokenizer_path: Path, cfg) -> dict:
    """Stream all cases once; small metadata stays in RAM and token IDs stay on disk."""
    if sys.byteorder != 'little':
        raise RuntimeError('Kaggle case caches use little-endian uint16')
    source = read_json(prepared / 'prepared_manifest.json')
    choices = {'prepared': source['identity'], 'tokenizer_sha256': sha256_file(tokenizer_path),
               'tokenizers': __import__('tokenizers').__version__, 'chunk_tokens': cfg.chunk_tokens}
    digest = identity(choices)
    root = Path(cfg.run_dir) / 'cache' / digest[:16]
    if (root / 'case_cache_manifest.json').is_file():
        return verify_cache(root, digest)
    # Reuse identical A/B caches from a saved profile/training run without copying bytes.
    for path in sorted(Path(cfg.input_root).rglob('case_cache_manifest.json')):
        if read_json(path).get('identity') == digest:
            return verify_cache(path.parent, digest)
    root.mkdir(parents=True, exist_ok=True)
    tokenizer = Tokenizer.from_file(str(tokenizer_path))
    if tokenizer.get_vocab_size() != 32768:
        raise ValueError('Use the matching own 32768-entry tokenizer')
    cases = {}
    token_count = 0
    counts = [0, 0]
''')


[Markdown Cell]

Importable source part 2

In [13]:
# [Code Cell]
append_module('data.py', r'''    capacity = cfg.chunk_tokens - 2
    with (root / 'tokens.bin.partial').open('wb') as stream:
        for split in ('train', 'dev', 'test'):
            current = []
            for row in tqdm(records(prepared / (split + '.jsonl')), total=source['counts'][split]['cases'], desc='Tokenize ' + split):
                encoded = tokenizer.encode(row['text'], add_special_tokens=False).ids
                if any(x < 0 or x >= 32768 for x in encoded):
                    raise ValueError('Tokenizer produced an invalid embedding ID')
                ensure_space(root, len(encoded) * 2, cfg.min_free_bytes)
                array('H', encoded).tofile(stream)
                normalized = re.sub(r'\s+', ' ', unicodedata.normalize('NFKC', row['text'])).strip()
                current.append({'id': row['id'], 'begin': token_count, 'length': len(encoded),
                                'chunks': max(1, math.ceil(len(encoded) / capacity)),
                                'gold': row.get('gold', -100), 'group': identity(normalized)})
                token_count += len(encoded)
                if split == 'train':
                    counts[row['gold']] += 1
            cases[split] = current
    (root / 'tokens.bin.partial').replace(root / 'tokens.bin')
    value = {'version': 'cjpe-cache-v1', 'identity': digest, 'choices': choices, 'cases': cases,
             'token_count': token_count, 'class_counts': counts, 'cases_dropped': 0,
             'all_document_tails_preserved': True,
             'artifacts': {'tokens.bin': {'sha256': sha256_file(root / 'tokens.bin'),
                                         'bytes': (root / 'tokens.bin').stat().st_size}}}
    value['manifest_sha256'] = identity(value)
    atomic_json(root / 'case_cache_manifest.json', value)
    return {**value, 'directory': str(root)}


def training_weights(cache: dict, cfg) -> np.ndarray:
    counts = np.asarray(cache['class_counts'], dtype=np.float64)
    if not (counts > 0).all():
        raise ValueError('Training needs both binary classes')
    enabled = cfg.class_weights == 'sqrt' or (cfg.class_weights == 'auto' and counts.max() / counts.min() >= 2)
    return np.clip(np.sqrt(counts.sum() / (2 * counts)), 0.25, 4.0).astype(np.float32) if enabled else np.ones(2, dtype=np.float32)


class CaseDataset:
    """One original case -> all chunks; labels remain separate from model inputs."""
    def __init__(self, cache: dict, split: str, cfg, teacher=None, representation=None):
        self.cache, self.split, self.cfg, self.teacher = cache, split, cfg, teacher
        self.policy = representation or cfg.representation
        self.rows = cache['cases'][split]
        self.tokens = np.memmap(Path(cache['directory']) / 'tokens.bin', dtype='<u2', mode='r') if cache['token_count'] else np.empty(0, dtype=np.uint16)

    def __len__(self):
        return len(self.rows)

''')


[Markdown Cell]

Importable source part 3

In [14]:
# [Code Cell]
append_module('data.py', r'''    def __getitem__(self, index):
        if index < 0:
            return {'chunks': [[1, 2]], 'positions': [[0., 0., 0.]], 'content_weights': [1.],
                    'gold': -100, 'index': -1, 'teacher': np.zeros(2, dtype=np.float32),
                    'kept_tokens': 0, 'total_tokens': 0}
        row = self.rows[index]
        capacity = self.cfg.chunk_tokens - 2
        indices = list(range(row['chunks']))
        if self.policy == 'first_last':
            count = self.cfg.section_chunks
            indices = sorted(set(indices[:count] + indices[-count:]))
        chunks, positions, weights = [], [], []
        for slot in indices:
            first = slot * capacity
            length = min(capacity, max(0, row['length'] - first))
            ids = self.tokens[row['begin'] + first:row['begin'] + first + length].astype(np.int64).tolist()
            chunks.append([1, *ids, 2])
            positions.append([slot / max(1, row['chunks'] - 1), math.log1p(row['chunks']) / 10, length / capacity])
            weights.append(float(max(1, length)))
        return {'chunks': chunks, 'positions': positions, 'content_weights': weights,
                'gold': row['gold'], 'index': index,
                'teacher': np.asarray(self.teacher[index] if self.teacher is not None else [0., 0.], dtype=np.float32),
                'kept_tokens': sum(len(x) - 2 for x in chunks), 'total_tokens': row['length']}

    def close(self):
        if getattr(self.tokens, '_mmap', None) is not None:
            self.tokens._mmap.close()


# CELL: Ragged token chunks and padded document sequences
def collate(rows: list[dict]) -> dict[str, torch.Tensor]:
    chunks = [chunk for row in rows for chunk in row['chunks']]
    width = max(map(len, chunks))
    count = max(len(r['chunks']) for r in rows)
    ids = torch.zeros(len(chunks), width, dtype=torch.long)
    mask = torch.zeros_like(ids, dtype=torch.bool)
    positions = torch.zeros(len(rows), count, 3)
    chunk_mask = torch.zeros(len(rows), count, dtype=torch.bool)
    content_weights = torch.zeros(len(rows), count)
    mapping = []
    first = 0
    for batch, row in enumerate(rows):
        length = len(row['chunks'])
        chunk_mask[batch, :length] = True
        positions[batch, :length] = torch.tensor(row['positions'])
        content_weights[batch, :length] = torch.tensor(row['content_weights'])
        for slot, chunk in enumerate(row['chunks']):
            ids[first, :len(chunk)] = torch.tensor(chunk)
''')


[Markdown Cell]

Importable source part 4

In [15]:
# [Code Cell]
append_module('data.py', r'''            mask[first, :len(chunk)] = True
            mapping.append(batch * count + slot)
            first += 1
    return {'input_ids': ids, 'attention_mask': mask, 'chunk_document': torch.tensor(mapping),
            'chunk_mask': chunk_mask, 'positions': positions, 'content_weights': content_weights,
            'targets': torch.tensor([r['gold'] for r in rows]),
            'indices': torch.tensor([r['index'] for r in rows]),
            'case_lengths': torch.tensor([len(r['chunks']) for r in rows], dtype=torch.long),
            'token_lengths': torch.tensor([len(chunk) for chunk in chunks], dtype=torch.long),
            'teacher': torch.from_numpy(np.stack([r['teacher'] for r in rows]))}


class CasePlan:
    """Length-bucketed shuffles; every case is owned once, with masked dummy tails."""
    def __init__(self, rows, cfg, rank, epoch):
        self.cfg, self.rank = cfg, rank
        self.global_size = cfg.world_size * cfg.micro_batch_size * cfg.accumulation_steps
        self.steps = math.ceil(len(rows) / self.global_size)
        generator = np.random.default_rng(cfg.seed + epoch)
        # Shuffle ties, then group similarly long cases into globally balanced updates.
        tie_order = generator.permutation(len(rows))
        ordered = sorted(tie_order, key=lambda i: rows[i]['chunks'])
        blocks = [ordered[i:i + self.global_size] for i in range(0, len(rows), self.global_size)]
        blocks = [blocks[i] for i in generator.permutation(len(blocks))]
        self.indices = np.concatenate([np.pad(generator.permutation(block), (0, self.global_size - len(block)), constant_values=-1) for block in blocks])

    def global_indices(self, step):
        return self.indices[step * self.global_size:(step + 1) * self.global_size]

    def micro_indices(self, step, micro):
        return self.global_indices(step).reshape(self.cfg.accumulation_steps, self.cfg.world_size, self.cfg.micro_batch_size)[micro, self.rank].tolist()
''')


[Markdown Cell]

### io_utils.py

Typed settings, strict serialization and verified own-foundation loading.

[Markdown Cell]

Importable source part 1

In [16]:
# [Code Cell]
append_module('io_utils.py', r'''# CELL: Strict JSON, checksums and atomic writes
from contextlib import contextmanager
import hashlib
import json
import os
from pathlib import Path
import shutil
import uuid

def strict_pairs(pairs: list[tuple[str, object]]) -> dict:
    result = {}
    for key, value in pairs:
        if key in result:
            raise ValueError(f'Duplicate JSON key: {key}')
        result[key] = value
    return result

def reject_constant(value: str):
    raise ValueError(f'Nonfinite JSON value: {value}')

def read_json(path: str | Path) -> dict:
    return json.loads(Path(path).read_text(encoding='utf-8'), object_pairs_hook=strict_pairs,
                      parse_constant=reject_constant)

def records(path: str | Path):
    with Path(path).open(encoding='utf-8') as stream:
        for line_number, line in enumerate(stream, 1):
            if line.strip():
                yield json.loads(line, object_pairs_hook=strict_pairs, parse_constant=reject_constant)

def sha256_file(path: str | Path) -> str:
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for chunk in iter(lambda: stream.read(8 << 20), b''):
            digest.update(chunk)
    return digest.hexdigest()

def identity(value: dict) -> str:
    return hashlib.sha256(json.dumps(value, sort_keys=True, separators=(',', ':'),
                                     allow_nan=False).encode()).hexdigest()

def atomic_json(path: str | Path, value: dict) -> None:
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + '.partial')
    with temporary.open('w', encoding='utf-8', newline='\n') as stream:
        json.dump(value, stream, ensure_ascii=False, indent=2, allow_nan=False)
        stream.write('\n'); stream.flush(); os.fsync(stream.fileno())
    os.replace(temporary, path)
''')


[Markdown Cell]

Importable source part 2

In [17]:
# [Code Cell]
append_module('io_utils.py', r'''
def ensure_space(directory: Path, needed: int, reserve: int) -> None:
    if shutil.disk_usage(directory).free < needed + reserve:
        raise OSError('Insufficient working disk. Start a fresh Kaggle session and attach the saved output; do not delete committed checkpoints.')

# CELL: Exclusive ownership and verified checkpoint copies
@contextmanager
def run_lock(directory: Path):
    """Linux advisory lock released by the OS after a killed Kaggle process."""
    import fcntl
    directory.mkdir(parents=True, exist_ok=True)
    with (directory / '.training.lock').open('a+') as stream:
        try:
            fcntl.flock(stream.fileno(), fcntl.LOCK_EX | fcntl.LOCK_NB)
        except BlockingIOError as error:
            raise RuntimeError('This experiment is already running. Do not launch a second training cell.') from error
        try:
            yield
        finally:
            fcntl.flock(stream.fileno(), fcntl.LOCK_UN)

def publish_copy(source: Path, destination: Path) -> None:
    """Publish a source without modifying it; hard-link only when supported."""
    if source.resolve() == destination.resolve():
        return
    temporary = destination.with_name(destination.name + '.' + uuid.uuid4().hex + '.partial')
    try:
        os.link(source, temporary)
    except OSError:
        shutil.copyfile(source, temporary)
    with temporary.open('r+b') as stream:
        os.fsync(stream.fileno())
    os.replace(temporary, destination)
''')


[Markdown Cell]

### legacy_contract.py

Exact known original runtime checksums for the optional full-state engine upgrade.

[Markdown Cell]

Importable source part 1

In [18]:
# [Code Cell]
append_module('legacy_contract.py', r'''LEGACY_SOURCE_SHA256 = {'__init__.py': '746816a2d6168979d1ab82a8ee3df4e15f9e1f02b5816e683799fd5723dec47c', 'adaptation.py': '62ecd0842a2c12254d4b1bb86ab100bd7392846c4f60c38b4cdc48b6321c1ad7', 'analysis.py': '36eaafc33d1f51dad9941c304f39fa6e525c5dd4bd804f75f8f002644f734be3', 'config.py': '2c3d310b92aef686c1128bc3b8c5d32211e7c966539dd52539ee3fa40613ac2f', 'data.py': '5af3ac4bab1123471d7cd3a0a2f832b97664dfeef5a6b26c07829ddd224fe1e9', 'io_utils.py': '3f9d828342c819690d390f0280c01f9eda160aab59c9cf424d6b3c2a31813e73', 'metrics.py': '8b1c074208775a9a64998567df8591e78045f3f5c38b99dce8bd0663e411a9db', 'model.py': '8fd795dfac04f4a1b05e2a21f91acb6b938280659288f2d7a2d140bd9e9a447f', 'parents.py': 'a41e6eacf9f053177c4e934ac547138a60d9cd1084e5f53948354a79b97047e9', 'predict.py': 'c1e089c389ce6240786bd80887275b9623e3baf2819b0c31c1590342d92c1333', 'preparation.py': '33a5e84c6576e6815eb53a767adcb8104f1aeb37f52a7d42e0d2d15f259bb0bf', 'prepare.py': '94d5cd8730c5439ce99c214e3007cee74ac4b6b1234279c40db926f05e253e47', 'profile.py': '7fe917c7a7d6fd9c453d76b9b4eeeb74610f7ccf7f0de8ab13793169229d1e9a', 'select.py': 'd0c4c1bc6731f5b311bcc9ac22dd9db6712da8fa54b5ad52f48bc0957aae306d', 'state.py': '369c7fd7d7e4ff16331f2f2f0a6348fd129c38c20bdbbd77fae786688bca0573', 'train.py': 'c76940a1b5a6a8828c1e30e4ba2a6eea138e2059f6a1af4429c86ab710ade001', 'workflow.py': '86f007570a1b301749b4ea4a7c6dd88677b4dca8643b26210839ca650b5f00cd'}

# Known first optimized engine; retained for verified full-state migration.
FAST_V1_SOURCE_SHA256 = {'__init__.py': '746816a2d6168979d1ab82a8ee3df4e15f9e1f02b5816e683799fd5723dec47c', 'adaptation.py': '62ecd0842a2c12254d4b1bb86ab100bd7392846c4f60c38b4cdc48b6321c1ad7', 'analysis.py': '36eaafc33d1f51dad9941c304f39fa6e525c5dd4bd804f75f8f002644f734be3', 'config.py': '08b73f7ed2e487398e51a66799a495e70051537473a5fdc3283da88371c8d073', 'data.py': '644725ec8027965a23754810ecd8c3e33018b1837b17af6f5deecbbca3c55a34', 'io_utils.py': '3f9d828342c819690d390f0280c01f9eda160aab59c9cf424d6b3c2a31813e73', 'legacy_contract.py': '66a0d0af191be23f4f00f46a16e7a04c7d3ef30f27e67908a78c1a26469e3a4a', 'metrics.py': '8b1c074208775a9a64998567df8591e78045f3f5c38b99dce8bd0663e411a9db', 'model.py': '4ff054235f673e1043f681dac4af0eb6713e36820079a56e02af39e1da9bd652', 'parents.py': 'a41e6eacf9f053177c4e934ac547138a60d9cd1084e5f53948354a79b97047e9', 'performance.py': 'bd3bfbaa476107db8c1896dbe5842055af2038dc0de36c8e427f0ee710f40ebf', 'predict.py': 'ab06d80b649a8f6fffbb8d6b8486ace44363bfc9cf02d0a0deada09db1697175', 'preparation.py': '33a5e84c6576e6815eb53a767adcb8104f1aeb37f52a7d42e0d2d15f259bb0bf', 'prepare.py': 'e2d10c17771adefb3c117963303483e2b5c6423bd4dae795d8238eda289012d5', 'profile.py': '7fe917c7a7d6fd9c453d76b9b4eeeb74610f7ccf7f0de8ab13793169229d1e9a', 'select.py': 'd0c4c1bc6731f5b311bcc9ac22dd9db6712da8fa54b5ad52f48bc0957aae306d', 'state.py': '369c7fd7d7e4ff16331f2f2f0a6348fd129c38c20bdbbd77fae786688bca0573', 'train.py': 'c31bce7a2e46658cde79f30c4a842f6f01486ea800c8ef86688a40701cb458c0', 'upgrade.py': '0c37b48e1356a4af3a973504c81b5a394f89582ac836c177c4871a5bcd469868', 'workflow.py': '86f007570a1b301749b4ea4a7c6dd88677b4dca8643b26210839ca650b5f00cd'}
''')


[Markdown Cell]

### metrics.py

Typed settings, strict serialization and verified own-foundation loading.

[Markdown Cell]

Importable source part 1

In [19]:
# [Code Cell]
append_module('metrics.py', r'''# CELL: Binary Macro-F1, strict original-ID output validation and official parity
import subprocess
import sys
from pathlib import Path
import numpy as np
from sklearn.model_selection import StratifiedGroupKFold
from .io_utils import atomic_json, read_json, records


def metrics(gold, predicted) -> dict:
    gold = np.asarray(gold, dtype=np.int64)
    predicted = np.asarray(predicted, dtype=np.int64)
    if gold.shape != predicted.shape or gold.ndim != 1 or not len(gold):
        raise ValueError('Gold/predicted class vectors must have identical nonempty shapes')
    if not np.isin(gold, [0, 1]).all() or not np.isin(predicted, [0, 1]).all():
        raise ValueError('Unknown binary outcome label')
    matrix = np.bincount(gold * 2 + predicted, minlength=4).reshape(2, 2)
    per_class = {}
    for index in range(2):
        true_positive = matrix[index, index]
        precision = true_positive / max(1, matrix[:, index].sum())
        recall = true_positive / max(1, matrix[index].sum())
        f1 = 2 * true_positive / max(1, matrix[index].sum() + matrix[:, index].sum())
        per_class[str(index)] = {'precision': float(precision), 'recall': float(recall),
                                 'f1': float(f1), 'support': int(matrix[index].sum())}
    return {'macro_f1': sum(x['f1'] for x in per_class.values()) / 2,
            'precision_positive': per_class['1']['precision'], 'recall_positive': per_class['1']['recall'],
            'accuracy': float((gold == predicted).mean()), 'cases': len(gold),
            'per_class': per_class, 'confusion_matrix': matrix.tolist()}


def probabilities(logits) -> np.ndarray:
    values = np.asarray(logits, dtype=np.float64)
    if values.ndim != 2 or values.shape[1] != 2 or not np.isfinite(values).all():
        raise ValueError('Case logits must be finite [N,2] arrays')
    shifted = values - values.max(1, keepdims=True)
    exp = np.exp(shifted)
    return exp[:, 1] / exp.sum(1)


def validate_prediction(predictions: list, rows: list) -> None:
    expected = [r['id'] for r in rows]
    if len(set(expected)) != len(expected) or not isinstance(predictions, list):
        raise ValueError('Invalid original IDs/output structure')
    actual = []
    for row in predictions:
        if (not isinstance(row, dict) or set(row) != {'id', 'pred'} or not isinstance(row['id'], str)
                or type(row['pred']) is not int or row['pred'] not in (0, 1)):
''')


[Markdown Cell]

Importable source part 2

In [20]:
# [Code Cell]
append_module('metrics.py', r'''            raise ValueError('Prediction must contain exactly original string id and integer pred=0/1')
        actual.append(row['id'])
    if actual != expected:
        raise ValueError('Missing/duplicate/reordered CJPE case IDs')


def prediction(probability, rows, threshold=0.5) -> list:
    probability = np.asarray(probability)
    if probability.shape != (len(rows),) or not np.isfinite(probability).all():
        raise ValueError('Case probability/ID lengths differ')
    result = [{'id': row['id'], 'pred': int(p >= threshold)} for row, p in zip(rows, probability)]
    validate_prediction(result, rows)
    return result


def official(prepared: Path, prediction_path: Path, expected: float, output: Path) -> None:
    truth = [{'id': r['id'], 'gold': r['gold']} for r in records(Path(prepared) / 'dev.jsonl')]
    atomic_json(Path(output) / 'dev_truth.json', truth)
    result = subprocess.run([sys.executable, str(Path(prepared) / 'evaluate_phase4.py'),
        '--truth', str(Path(output) / 'dev_truth.json'), '--prediction', str(prediction_path)],
        capture_output=True, text=True, check=True)
    line = next(line for line in result.stdout.splitlines() if line.startswith('Macro-F1:'))
    if abs(float(line.split(':')[1]) - expected) > 1e-6:
        raise ValueError('Native metric differs from official evaluate_phase4.py')
    (Path(output) / 'official_dev_evaluation.txt').write_text(result.stdout, encoding='utf-8')


# CELL: Small predeclared threshold grid, grouped conditional validation and fallback
def choose_threshold(probability, rows, enabled=True, margin=0.002, seed=781) -> dict:
    gold = np.asarray([r['gold'] for r in rows], dtype=np.int64)
    probability = np.asarray(probability, dtype=np.float64)
    grid = np.round(np.arange(0.2, 0.801, 0.05), 2)
    def choose(indices):
        scores = [metrics(gold[indices], (probability[indices] >= x).astype(int))['macro_f1'] for x in grid]
        return max(range(len(grid)), key=lambda i: (scores[i], -abs(grid[i] - .5))), scores
    full = np.arange(len(rows))
    chosen, scores = choose(full)
    baseline = metrics(gold, (probability >= .5).astype(int))['macro_f1']
    groups = np.asarray([r.get('group', r['id']) for r in rows])
    folds = min(5, int(np.bincount(gold, minlength=2).min()), len(set(groups)))
    conditional = None
    selected_thresholds = []
    if enabled and folds >= 2:
        out_of_fold = np.zeros(len(rows), dtype=np.int64)
        valid = True
        for train, heldout in StratifiedGroupKFold(folds, shuffle=True, random_state=seed).split(full, gold, groups):
            if len(set(gold[train])) < 2 or not len(heldout):
                valid = False
''')


[Markdown Cell]

Importable source part 3

In [21]:
# [Code Cell]
append_module('metrics.py', r'''                break
            index, _ = choose(train)
            threshold = float(grid[index])
            out_of_fold[heldout] = probability[heldout] >= threshold
            selected_thresholds.append(threshold)
        if valid:
            conditional = metrics(gold, out_of_fold)['macro_f1']
    improved = (enabled and conditional is not None and scores[chosen] >= baseline + margin
                and conditional >= baseline + margin)
    threshold = float(grid[chosen]) if improved else .5
    return {'threshold': threshold, 'calibration_selected': bool(improved), 'grid': grid.tolist(),
            'raw_dev_macro_f1': baseline, 'selected_dev_macro_f1': metrics(gold, (probability >= threshold).astype(int))['macro_f1'],
            'best_fitted_threshold': float(grid[chosen]), 'best_fitted_dev_macro_f1': scores[chosen],
            'conditional_threshold_OOF_macro_f1': conditional, 'fold_thresholds': selected_thresholds,
            'minimum_gain': margin, 'scope': 'Conditional on checkpoints already selected on this dev set, not an independent test estimate',
            'test_labels_used': False}


def case_diagnostics(gold, predicted, rows) -> dict:
    result = metrics(gold, predicted)
    result['errors'] = [{'id': r['id'], 'gold': int(y), 'pred': int(p), 'tokens': r.get('length')}
                        for r, y, p in zip(rows, gold, predicted) if y != p][:50]
    result['length_buckets'] = {}
    lengths = np.asarray([r.get('length', 0) for r in rows])
    gold, predicted = np.asarray(gold), np.asarray(predicted)
    for lo, hi in ((0, 512), (512, 2048), (2048, 8192), (8192, float('inf'))):
        selected = (lengths >= lo) & (lengths < hi)
        if selected.any():
            result['length_buckets'][str(lo) + ':' + str(hi)] = metrics(gold[selected], predicted[selected])
    return result
''')


[Markdown Cell]

### model.py

Identical model tensors and case objective. CPU pack metadata trims only padding. Keep active encoder checkpointing, move the tiny projection outside it, and checkpoint only the trainable D suffix after a graph-free frozen prefix. C adapters/B BitFit still need gradients through frozen intermediate layers.

[Markdown Cell]

Importable source part 1

In [22]:
# [Code Cell]
append_module('model.py', r'''# CELL: Token features -> bounded chunk projections -> one hierarchical case prediction
from dataclasses import replace
from pathlib import Path
import torch
from torch import nn
from torch.nn import functional as F
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
from torch.utils.checkpoint import checkpoint
from .adaptation import adapt
from .parents import parent_class


class CaseClassifier(nn.Module):
    """Ragged chunks [P,T] -> padded document chunks [B,C,H] -> logits [B,2].

    All full-document chunks are represented. Chunk microbatching and recomputation
    bound encoder activation memory without treating chunks as independent cases.
    """
    def __init__(self, backbone: nn.Module, width: int, cfg):
        super().__init__()
        self.backbone, self.cfg = backbone, cfg
        self.projection = nn.Sequential(nn.Linear(width * 2, cfg.head_width), nn.GELU(), nn.LayerNorm(cfg.head_width))
        self.positions = nn.Linear(3, cfg.head_width, bias=False)
        self.context = nn.GRU(cfg.head_width, cfg.head_width // 2, batch_first=True, bidirectional=True) if cfg.recurrent_chunks else None
        self.norm = nn.LayerNorm(cfg.head_width)
        self.attention = nn.Sequential(nn.Linear(cfg.head_width, cfg.head_width // 2), nn.Tanh(), nn.Linear(cfg.head_width // 2, 1)) if cfg.pooling == 'attention' else None
        self.dropout = nn.Dropout(cfg.dropout)
        self.classifier = nn.Linear(cfg.head_width * 2, 2)

    def hidden_states(self, ids: torch.Tensor, mask: torch.Tensor, full: bool) -> torch.Tensor:
        """Same own feature path; padding is verified on CPU, avoiding GPU bool sync."""
        if self.cfg.family == 'b':
            return self.backbone.encode(ids, mask)
        if full:
            return self.backbone.hidden_states(ids, None)
        if self.cfg.family == 'd':
            hidden = self.backbone.token_embedding(ids)
            for layer in self.backbone.layers:
                hidden = layer(hidden, mask)
        else:
            causal = torch.ones((ids.shape[1], ids.shape[1]), device=ids.device, dtype=torch.bool).tril()
            allowed = causal[None, None] & mask[:, None, None, :]
            hidden = self.backbone.token_embedding(ids)
            modules = self.backbone.stages if self.cfg.family == 'c' else self.backbone.layers
            for layer in modules:
                hidden = layer(hidden, allowed)
        return self.backbone.final_norm(hidden)

''')


[Markdown Cell]

Importable source part 2

In [23]:
# [Code Cell]
append_module('model.py', r'''    @staticmethod
    def pool(hidden: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
        """Token features [p,T,D] -> FP32 mean/last [p,2D], excluding padding."""
        hidden = hidden.float()
        lengths = mask.sum(1).clamp_min(1)
        mean = (hidden * mask[..., None]).sum(1) / lengths[:, None]
        last = hidden[torch.arange(hidden.shape[0], device=hidden.device), lengths - 1]
        return torch.cat((mean, last), dim=-1)

    def encoder_pool(self, ids, mask, full):
        return self.pool(self.hidden_states(ids, mask, full), mask)

    def suffix_pool(self, hidden, mask):
        for layer in self.backbone.layers[-self.cfg.partial_layers:]:
            hidden = layer(hidden, mask)
        return self.pool(self.backbone.final_norm(hidden), mask)

    def chunk_features(self, ids: torch.Tensor, mask: torch.Tensor, full=False, cache_prefix=True) -> torch.Tensor:
        """Checkpoint only active encoder work; never repeat D's frozen prefix."""
        active_checkpoint = self.training and self.cfg.checkpoint_encoder
        if self.cfg.family == 'd' and self.cfg.method == 'partial' and cache_prefix:
            hidden = self.backbone.token_embedding(ids)
            valid = None if full else mask
            # Every prefix parameter is frozen and IDs are integers: no autograd
            # graph is created. Keep training mode/dropout and sparse route cache.
            for layer in self.backbone.layers[:-self.cfg.partial_layers]:
                hidden = layer(hidden, valid)
            assert not hidden.requires_grad, 'Frozen D prefix unexpectedly became trainable'
            # Pool mask still excludes right padding; attention may use None only
            # when every token is valid. No detach is applied to trainable suffixes.
            if active_checkpoint:
                pooled = checkpoint(self.suffix_pool, hidden, mask, use_reentrant=False)
            else:
                pooled = self.suffix_pool(hidden, mask)
        elif active_checkpoint:
            pooled = checkpoint(self.encoder_pool, ids, mask, full, use_reentrant=False)
        else:
            pooled = self.encoder_pool(ids, mask, full)
        return self.projection(pooled)

    def forward(self, input_ids: torch.Tensor, attention_mask: torch.Tensor,
                chunk_document: torch.Tensor, chunk_mask: torch.Tensor,
                positions: torch.Tensor, content_weights: torch.Tensor,
                return_attention: bool = False, case_lengths=None, pack_layout=None, cache_prefix=True):
        features = []
        if pack_layout is None:
            lengths = attention_mask.sum(1).cpu().tolist()
            pack_layout = make_pack_layout(lengths, self.cfg.chunk_micro_batch)
''')


[Markdown Cell]

Importable source part 3

In [24]:
# [Code Cell]
append_module('model.py', r'''        for first, stop, width, full in pack_layout:
            features.append(self.chunk_features(input_ids[first:stop, :width], attention_mask[first:stop, :width], full, cache_prefix))
        batch, count = chunk_mask.shape
        flat = torch.zeros(batch * count, self.cfg.head_width, device=input_ids.device, dtype=torch.float32)
        flat = flat.index_copy(0, chunk_document, torch.cat(features).float())
        hidden = flat.reshape(batch, count, -1) + self.positions(positions)
        if self.context is not None:
            # DDP moves tensor arguments to its rank's CUDA device, including
            # metadata. Always construct this tensor on CPU for the packed GRU.
            if case_lengths is None:
                lengths = chunk_mask.sum(1).to(device='cpu', dtype=torch.long)
            elif isinstance(case_lengths, torch.Tensor):
                lengths = case_lengths.to(device='cpu', dtype=torch.long)
            else:
                lengths = torch.tensor(case_lengths, device='cpu', dtype=torch.long)
            packed = pack_padded_sequence(hidden, lengths, batch_first=True, enforce_sorted=False)
            packed, _ = self.context(packed)
            context, _ = pad_packed_sequence(packed, batch_first=True, total_length=count)
            hidden = hidden + context
        hidden = self.norm(hidden)
        mean_weights = content_weights * chunk_mask
        mean_weights = mean_weights / mean_weights.sum(1, keepdim=True).clamp_min(1)
        if self.attention is None or self.cfg.pooling == 'mean':
            weights = mean_weights
        else:
            scores = self.attention(hidden).squeeze(-1).float().masked_fill(~chunk_mask, float('-inf'))
            weights = torch.softmax(scores, dim=1)
        pooled = (hidden * weights[..., None]).sum(1)
        mean = (hidden * mean_weights[..., None]).sum(1)
        logits = self.classifier(self.dropout(torch.cat((pooled, mean), dim=-1)))
        return (logits, weights) if return_attention else logits


def forward(model, batch: dict, device: torch.device, return_attention=False):
    names = ('input_ids', 'attention_mask', 'chunk_document', 'chunk_mask', 'positions', 'content_weights')
    actual = model.module if hasattr(model, 'module') else model
    lengths = (batch['token_lengths'] if 'token_lengths' in batch else batch['attention_mask'].sum(1)).tolist()
    layout = make_pack_layout(lengths, actual.cfg.chunk_micro_batch)
    counts = batch['case_lengths'] if 'case_lengths' in batch else batch['chunk_mask'].sum(1)
    # Python scalars survive DDP's recursive CUDA transfer unchanged. This also
    # avoids moving CPU lengths to CUDA and synchronizing them back on every call.
    case_lengths = tuple(int(value) for value in counts.tolist())
    return model(**{name: batch[name].to(device, non_blocking=True) for name in names},
                 return_attention=return_attention, case_lengths=case_lengths, pack_layout=layout,
                 cache_prefix=sum(lengths) <= actual.cfg.frozen_prefix_cache_tokens)


def make_pack_layout(lengths: list[int], micro_batch: int) -> tuple:
''')


[Markdown Cell]

Importable source part 4

In [25]:
# [Code Cell]
append_module('model.py', r'''    """CPU-only contiguous packs: trim padding, preserve token and chunk order."""
    result = []
    for first in range(0, len(lengths), micro_batch):
        stop = min(len(lengths), first + micro_batch)
        current = lengths[first:stop]
        width = max(current)
        result.append((first, stop, width, min(current) == width))
    return tuple(result)


def build_model(job: dict, cfg, device: torch.device, load_parent=True) -> CaseClassifier:
    Config, Model = parent_class(job['parent_source_dir'], cfg.family, job['parent_contract'])
    changes = {'activation_checkpointing': False}
    if cfg.family == 'd':
        changes['sparse_attention_checkpointing'] = False
    base_cfg = replace(Config(**job['parent']['config']), **changes)
    with torch.device('meta'):
        backbone = Model(base_cfg)
    if load_parent:
        path = Path(job['parent']['checkpoint'])
        if path.stat().st_size != job['parent']['checkpoint_bytes'] or path.stat().st_mtime_ns != job['parent']['checkpoint_mtime_ns']:
            raise ValueError('Verified foundation checkpoint changed before loading')
        payload = torch.load(path, map_location='cpu', weights_only=True, mmap=True)
        backbone.load_state_dict(payload['model'], strict=True, assign=True)
        del payload
    else:
        backbone._apply(lambda tensor: torch.empty_like(tensor, device=device) if tensor.is_meta else tensor.to(device))
    foundation = sum(p.numel() for p in backbone.parameters())
    assert 50_000_000 < foundation < 300_000_000
    capacity = base_cfg.max_source_tokens if cfg.family == 'b' else base_cfg.max_seq_tokens
    if cfg.chunk_tokens > capacity:
        raise ValueError('chunk_tokens exceeds the actual foundation context capacity')
    changed = adapt(backbone, cfg)
    with torch.random.fork_rng(devices=[]):
        torch.random.default_generator.manual_seed(cfg.seed + 1000)
        model = CaseClassifier(backbone, base_cfg.d_model, cfg).to(device)
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    assert 50_000_000 < total < 300_000_000 and trainable > 0
    model.parameter_report = {'foundation_unique_parameters': foundation, 'total_unique_parameters': total,
        'trainable_parameters': trainable, 'active_backbone_trainable_parameters': sum(p.numel() for p in backbone.parameters() if p.requires_grad),
        'adapted_names': changed, 'inactive_B_decoder': cfg.family == 'b'}
    return model
''')


[Markdown Cell]

### parents.py

Typed settings, strict serialization and verified own-foundation loading.

[Markdown Cell]

Importable source part 1

In [26]:
# [Code Cell]
append_module('parents.py', r'''# CELL: Match a parent by source and data rather than a fixed epoch count
import hashlib
import importlib
import importlib.util
import json
import math
from pathlib import Path
import sys
import torch
from tokenizers import Tokenizer
from .io_utils import atomic_json, read_json, sha256_file

def source_hashes(directory: Path) -> dict[str,str]:
    return {path.name: sha256_file(path) for path in sorted(directory.glob('*.py'))}

def learning_choices(values: dict, contract: dict) -> dict:
    return {key:value for key,value in values.items() if key not in contract['fingerprint_exclusions']}

def reproduce_parent(values: dict, contract: dict, version: str) -> str:
    payload={'version':contract['runtime_version'],'config':learning_choices(values,contract),
             'data':contract['data_fingerprint'],'sources':contract['source_sha256'],'torch':version}
    return hashlib.sha256(json.dumps(payload,sort_keys=True).encode()).hexdigest()

def intact_pairs(root: Path) -> tuple[list, list]:
    """Check receipts/checkpoint bytes incrementally before mmap loading."""
    pairs=[]; warnings=[]
    for name in ('slot_0.pt','slot_1.pt','best.pt'):
        path=root/'checkpoints'/name; receipt_path=path.with_name(name+'.json')
        if not path.exists() and not receipt_path.exists():
            continue
        try:
            receipt=read_json(receipt_path)
            if any(type(receipt.get(key)) is not int or receipt[key]<0 for key in ('bytes','epoch','step','generation')):
                raise ValueError('Invalid receipt counters')
            if receipt['bytes']!=path.stat().st_size or receipt['sha256']!=sha256_file(path):
                raise ValueError('Size/SHA-256 mismatch')
            pairs.append((path,receipt))
        except (ValueError,OSError,KeyError,TypeError) as error:
            warnings.append(f'{path}: {error}')
    if not pairs:
        raise FileNotFoundError('No intact full checkpoint/receipt pair. ' + '; '.join(warnings))
    return pairs,warnings

def parent_candidates(cfg, contract: dict) -> list[dict]:
    """Return useful rejection reasons for every discovered run root."""
    roots=[Path(cfg.parent_run_dir)] if cfg.parent_run_dir else sorted({p.parent for p in Path(cfg.input_root).rglob('train_config.json')})
    rows=[]
    for root in roots:
''')


[Markdown Cell]

Importable source part 2

In [27]:
# [Code Cell]
append_module('parents.py', r'''        row={'run_dir':str(root),'reasons':[],'accepted':False}
        try:
            required=('train_config.json','data_report.json','training_summary.json',f'code/model_{cfg.family}/model.py')
            missing=[name for name in required if not (root/name).is_file()]
            if missing:
                row['reasons'].append('Missing: '+', '.join(missing))
            else:
                saved=read_json(root/'train_config.json'); summary=read_json(root/'training_summary.json')
                row.update(planned_epochs=saved.get('epochs'),completed_epochs=summary.get('epoch'),step=summary.get('step'),status=summary.get('status'))
                if type(saved.get('epochs')) is not int or saved['epochs']<1:
                    row['reasons'].append('Invalid configured epoch count')
                actual=source_hashes(root/f'code/model_{cfg.family}')
                changed=sorted(name for name in set(actual)|set(contract['source_sha256']) if actual.get(name)!=contract['source_sha256'].get(name))
                if changed:
                    row['reasons'].append('Original source differs: '+', '.join(changed))
                if read_json(root/'data_report.json').get('data_fingerprint')!=contract['data_fingerprint']:
                    row['reasons'].append('Wrong preprocessing identity for this model')
                names=[name for name in ('slot_0.pt','slot_1.pt','best.pt') if (root/'checkpoints'/name).is_file()
                       and (root/'checkpoints'/(name+'.json')).is_file()]
                row['checkpoint_pairs']=names
                if not names:
                    row['reasons'].append('Attach saved output with real .pt files and .pt.json receipts; .ipynb/prediction ZIP is insufficient')
                row['accepted']=not row['reasons']
        except (ValueError,OSError,KeyError,TypeError,AttributeError) as error:
            row['reasons'].append(f'{type(error).__name__}: {error}')
        rows.append(row)
    return rows

# CELL: Check real cursor/config/Torch identity before selecting weights
def validate_payload(payload: dict, receipt: dict, values: dict, contract: dict) -> None:
    required={'model','config','cursor','optimizer','scaler','scheduler','rng_by_rank','world_size'}
    if required-payload.keys():
        raise ValueError('Parent checkpoint lacks full recovery state')
    cursor=payload['cursor']
    if (any(type(cursor.get(key)) is not int or cursor[key]<0 for key in ('epoch','next_batch','global_step','optimizer_steps'))
            or payload['fingerprint']!=receipt['fingerprint'] or payload['step']!=receipt['step']
            or payload['epoch']!=receipt['epoch'] or payload['epoch']!=cursor['epoch']
            or payload['step']!=cursor['global_step'] or cursor['epoch']>values['epochs']
            or cursor['optimizer_steps']>cursor['global_step']
            or payload['scheduler']['steps']!=cursor['optimizer_steps']
            or payload['world_size']!=values['world_size'] or len(payload['rng_by_rank'])!=values['world_size']
            or learning_choices(payload['config'],contract)!=learning_choices(values,contract)):
        raise ValueError('Parent cursor/config/scheduler/RNG/receipt disagree')

def verify_parent(cfg, contract: dict) -> tuple[Path,dict]:
    diagnostics=parent_candidates(cfg,contract)
    atomic_json(Path(cfg.run_dir)/'parent_input_report.json',{'candidates':diagnostics})
    print(json.dumps({'parent_candidates':diagnostics},indent=2),flush=True)
''')


[Markdown Cell]

Importable source part 3

In [28]:
# [Code Cell]
append_module('parents.py', r'''    accepted=[Path(row['run_dir']) for row in diagnostics if row['accepted']]
    if len(accepted)!=1:
        raise ValueError(f'Found {len(accepted)} matching Model {cfg.family.upper()} outputs. Attach ONE full saved '
                         'output or set parent_run_dir to its exact printed run_dir; see parent_input_report.json.')
    root=accepted[0]; values=read_json(root/'train_config.json')
    pairs,warnings=intact_pairs(root)
    latest=max(pairs,key=lambda item:(item[1]['step'],item[1]['generation']))
    payload=torch.load(latest[0],map_location='cpu',weights_only=True,mmap=True)
    validate_payload(payload,latest[1],values,contract)
    versions=list(contract['recorded_torch_candidates'])+[str(torch.__version__)]
    for name in ('continuation_runtime_identity.json','training_runtime_identity.json'):
        path=root/name
        if path.is_file():
            record=read_json(path)
            if isinstance(record.get('torch'),str):
                versions.insert(0,record['torch'])
    matched=next((v for v in dict.fromkeys(versions) if reproduce_parent(values,contract,v)==payload['fingerprint']),None)
    if matched is None:
        raise ValueError('Cannot reproduce original config/source/data/Torch fingerprint. Attach the matching original runtime and tokenizer; do not disable this check.')
    cursor=payload['cursor']
    if cursor['epoch']!=values['epochs'] or cursor['next_batch']!=0:
        raise ValueError(f"Parent has {cursor['epoch']}/{values['epochs']} completed epochs, next_batch={cursor['next_batch']}. Finish/save its continuation before starting a new CJPE experiment.")
    report=read_json(root/'data_report.json')
    examples=report.get('train_examples')
    coverage=(isinstance(report.get('eligible_source_tokens'),dict) if cfg.family=='b' else report.get('full_window_coverage') is True)
    if type(examples) is not int or examples<1 or not coverage:
        raise ValueError('Parent data report lacks a positive exhaustive training-window count')
    global_batch=values['world_size']*values['micro_batch_size']*values['accumulation_steps']
    expected_steps=math.ceil(examples/global_batch)*cursor['epoch']
    if cursor['global_step']!=expected_steps:
        raise ValueError(f"Parent counters disagree with its exhaustive window plan: expected {expected_steps} boundaries, got {cursor['global_step']}")
    summary=read_json(root/'training_summary.json')
    if summary.get('status')!='complete' or summary.get('epoch')!=cursor['epoch'] or summary.get('step')!=cursor['global_step'] or summary.get('fingerprint')!=payload['fingerprint']:
        raise ValueError('Parent training summary disagrees with its latest intact checkpoint')
    chosen=next((item for item in pairs if item[0].name=='best.pt'),latest) if cfg.parent_checkpoint=='best' else latest
    selected=torch.load(chosen[0],map_location='cpu',weights_only=True,mmap=True)
    validate_payload(selected,chosen[1],values,contract)
    if selected['fingerprint']!=payload['fingerprint']:
        raise ValueError('Selected parent best/latest belongs to another run')
    for name,tensor in selected['model'].items():
        if tensor.is_floating_point():
            if tensor.dtype!=torch.float32 or not all(bool(torch.isfinite(c).all()) for c in tensor.reshape(-1).split(1_000_000)):
                raise ValueError(f'Nonfinite/non-FP32 master weights: {name}')
    result={'family':cfg.family,'config':values,'completed_epochs':cursor['epoch'],'planned_epochs':values['epochs'],
            'checkpoint':str(chosen[0]),'checkpoint_sha256':chosen[1]['sha256'],'checkpoint_bytes':chosen[1]['bytes'],
            'checkpoint_mtime_ns':chosen[0].stat().st_mtime_ns,'checkpoint_epoch':chosen[1]['epoch'],'checkpoint_step':chosen[1]['step'],
            'selection':cfg.parent_checkpoint,'original_fingerprint':payload['fingerprint'],'original_torch':matched,
            'data_fingerprint':contract['data_fingerprint'],'source_sha256':contract['source_sha256'],'checkpoint_warnings':warnings}
''')


[Markdown Cell]

Importable source part 4

In [29]:
# [Code Cell]
append_module('parents.py', r'''    print(f"Verified Model {cfg.family.upper()}: {cursor['epoch']} completed epochs; selected {chosen[0].name} at epoch {chosen[1]['epoch']}",flush=True)
    return root,result

# CELL: Find exactly matching small preprocessing artifacts without reading LM binaries
def tokenizer_inputs(cfg, contract: dict) -> tuple[Path,Path,dict]:
    metadata=[Path(cfg.metadata_path)] if cfg.metadata_path else sorted(Path(cfg.input_root).rglob('metadata.json'))
    matches=[]
    required=('train.bin','val.bin','tokenizer.json','train_documents.jsonl','validation_documents.jsonl','validation_pairs.jsonl')
    for path in metadata:
        if not path.is_file():
            continue
        try:
            value=read_json(path)
            hashes={name:value['artifacts'][name]['sha256'] for name in required}
            hashes['metadata.json']=sha256_file(path)
            fingerprint=hashlib.sha256(json.dumps(hashes,sort_keys=True).encode()).hexdigest()
            if fingerprint==contract['data_fingerprint']:
                matches.append((path,value))
        except (KeyError,TypeError,ValueError,OSError):
            continue
    if not matches:
        raise FileNotFoundError(f'Missing matching Model {cfg.family.upper()} metadata.json. Use its original preprocessing output, not another token-budget export; large .bin files are unnecessary.')
    meta,value=matches[0]; expected=value['artifacts']['tokenizer.json']['sha256']
    candidates=[Path(cfg.tokenizer_path)] if cfg.tokenizer_path else [meta.parent/'tokenizer.json',*Path(cfg.input_root).rglob('tokenizer.json')]
    token=next((p for p in candidates if p.is_file() and sha256_file(p)==expected),None)
    if token is None:
        raise FileNotFoundError('No tokenizer.json matches the verified metadata; attach the original tokenizer')
    tokenizer=Tokenizer.from_file(str(token))
    specials={'<pad>':0,'<s>':1,'</s>':2,'<unk>':3,'<sep>':4,'<mask>':5}
    if tokenizer.get_vocab_size()!=32768 or any(tokenizer.token_to_id(k)!=v for k,v in specials.items()):
        raise ValueError('Unexpected parent tokenizer vocabulary/special IDs')
    return token,meta,{'tokenizer_sha256':expected,'metadata_sha256':sha256_file(meta)}

def parent_class(directory: str, family: str, contract: dict):
    """Import only the checksum-verified original config and model implementation."""
    root=Path(directory)
    if source_hashes(root)!=contract['source_sha256']:
        raise ValueError('Parent source changed after verification')
    package='verified_parent_'+family+'_'+contract['source_sha256']['model.py'][:12]
    if package not in sys.modules:
        spec=importlib.util.spec_from_file_location(package,root/'__init__.py',submodule_search_locations=[str(root)])
        module=importlib.util.module_from_spec(spec); sys.modules[package]=module; spec.loader.exec_module(module)
    config=importlib.import_module(package+'.config'); model=importlib.import_module(package+'.model')
    return config.TrainConfig,getattr(model,contract['class_name'])
''')


[Markdown Cell]

### performance.py

Bounded one-batch-ahead CPU collation/pinning with no stochastic augmentation. Pending rows are regenerated from the saved cursor; training rows and chunks are never removed.

[Markdown Cell]

Importable source part 1

In [30]:
# [Code Cell]
append_module('performance.py', r'''# CELL: Bounded deterministic CPU prefetch; no worker RNG or data sampling
from concurrent.futures import ThreadPoolExecutor
import torch
from .data import collate


class MicroBatches:
    """At most one CPU/pinned batch ahead of the exact saved row cursor.

    Collation uses no random transforms, device work or model state. Pending rows
    are discarded on a stop and regenerated from the same cursor on recovery.
    """
    def __init__(self, dataset, plan, first_step: int, cfg, pin_memory=True):
        self.dataset, self.cfg = dataset, cfg
        self.pin_memory = pin_memory and torch.cuda.is_available()
        self.indices = (plan.micro_indices(step, micro) for step in range(first_step, plan.steps)
                        for micro in range(cfg.accumulation_steps))
        self.executor = ThreadPoolExecutor(max_workers=1) if cfg.prefetch_batches else None
        self.pending = None

    def build(self, indices):
        batch = collate([self.dataset[index] for index in indices])
        if self.pin_memory:
            batch = {name: value.pin_memory() if name not in ('case_lengths', 'token_lengths', 'indices') else value
                     for name, value in batch.items()}
        return batch

    def __enter__(self):
        if self.executor:
            indices = next(self.indices, None)
            if indices is not None:
                self.pending = self.executor.submit(self.build, indices)
        return self

    def __next__(self):
        if self.executor is None:
            return self.build(next(self.indices))
        if self.pending is None:
            raise StopIteration
        batch = self.pending.result()
        indices = next(self.indices, None)
        self.pending = self.executor.submit(self.build, indices) if indices is not None else None
        return batch

    def __exit__(self, *_):
        if self.executor:
            if self.pending:
                self.pending.cancel()
''')


[Markdown Cell]

Importable source part 2

In [31]:
# [Code Cell]
append_module('performance.py', r'''            self.executor.shutdown(wait=True)
''')


[Markdown Cell]

### predict.py

Two GPU inference replicas balance all cases by chunk work and batch similar lengths. Export train/dev before freezing the threshold; only then process test. No rows are truncated to the inference chunk budget: a longer case is processed alone.

[Markdown Cell]

Importable source part 1

In [32]:
# [Code Cell]
append_module('predict.py', r'''# CELL: Full-case inference and strict learned-task reconstruction
import argparse
from dataclasses import replace
from pathlib import Path
import zipfile
import numpy as np
import torch
import torch.multiprocessing as mp
from tqdm.auto import tqdm
from .config import CJPEConfig
from .data import CaseDataset, cache_data, collate, verify_cache
from .io_utils import atomic_json, read_json, sha256_file
from .metrics import case_diagnostics, choose_threshold, metrics, official, prediction, probabilities, validate_prediction
from .model import build_model, forward
from .parents import source_hashes
from .preparation import prepared_input


def logits_partition(model, job, cfg, split, device, rank=0, world_size=1, representation=None):
    dataset = CaseDataset(job['cache'], split, cfg, representation=representation)
    output = {}
    indices = balanced_indices(dataset.rows, rank, world_size)
    batches, current, work = [], [], 0
    for index in indices:
        count = dataset.rows[index]['chunks']
        if current and (len(current) >= cfg.inference_cases_per_batch or work + count > cfg.inference_chunk_budget):
            batches.append(current)
            current, work = [], 0
        current.append(index)
        work += count
    if current:
        batches.append(current)
    model.eval()
    progress = tqdm(total=len(indices), desc='Predict ' + split, disable=rank != 0, unit='cases')
    with torch.inference_mode():
        for selected in batches:
            batch = collate([dataset[index] for index in selected])
            with torch.autocast(device.type, dtype=torch.float16, enabled=device.type == 'cuda'):
                logits = forward(model, batch, device)
            values = logits.float().cpu().numpy()
            if values.shape != (len(selected), 2) or not np.isfinite(values).all():
                raise ValueError('Nonfinite/wrong-shaped case logits')
            output.update(zip(selected, values))
            progress.update(len(selected))
    progress.close()
    dataset.close()
    return output

''')


[Markdown Cell]

Importable source part 2

In [33]:
# [Code Cell]
append_module('predict.py', r'''
def balanced_indices(rows, rank, world_size):
    """Assign all cases once by estimated chunk work, then batch similar lengths."""
    assigned = [[] for _ in range(world_size)]
    work = [0] * world_size
    for index in sorted(range(len(rows)), key=lambda i: (-rows[i]['chunks'], i)):
        owner = min(range(world_size), key=lambda r: (work[r], r))
        assigned[owner].append(index)
        work[owner] += rows[index]['chunks']
    return sorted(assigned[rank], key=lambda i: (rows[i]['chunks'], i))


def inference_worker(rank, job_path, splits):
    """Independent GPU replicas for inference; training still uses NCCL DDP."""
    torch.set_num_threads(2)
    torch.cuda.set_device(rank)
    job = read_json(job_path)
    root = Path(job['config']['run_dir'])
    model, local, cfg, _ = load_task(root, Path(job['prepared_dir']), torch.device('cuda', rank), root / 'inference_cache' / str(rank))
    directory = root / 'predictions'
    directory.mkdir(exist_ok=True)
    for split in splits:
        values = logits_partition(model, local, cfg, split, torch.device('cuda', rank), rank, 2)
        indices = sorted(values)
        logits = np.stack([values[i] for i in indices]) if indices else np.empty((0, 2), dtype=np.float32)
        path = directory / (split + f'_rank{rank}.npz')
        with path.with_suffix('.npz.partial').open('wb') as stream:
            np.savez(stream, indices=np.asarray(indices, dtype=np.int64), logits=logits)
        path.with_suffix('.npz.partial').replace(path)
    atomic_json(directory / f'parameters_rank{rank}.json', model.parameter_report)


def parallel_predict(job_path, splits):
    """Merge exact original-row order only after both complete GPU shards exist."""
    job = read_json(job_path)
    directory = Path(job['config']['run_dir']) / 'predictions'
    mp.spawn(inference_worker, args=(str(job_path), tuple(splits)), nprocs=2, join=True)
    for split in splits:
        values = {}
        for rank in range(2):
            with np.load(directory / (split + f'_rank{rank}.npz'), allow_pickle=False) as shard:
                for index, logits in zip(shard['indices'], shard['logits']):
                    if int(index) in values:
                        raise ValueError('Inference ranks duplicated a case')
                    values[int(index)] = logits
        logits = ordered_logits(values, job['cache']['cases'][split])
        np.save(directory / (split + '_logits.npy'), logits)
        atomic_json(directory / (split + '_order.json'), [{'id': r['id']} for r in job['cache']['cases'][split]])
''')


[Markdown Cell]

Importable source part 3

In [34]:
# [Code Cell]
append_module('predict.py', r'''    return read_json(directory / 'parameters_rank0.json')


def ordered_logits(values, rows):
    if set(values) != set(range(len(rows))):
        raise ValueError('Inference must cover each original case exactly once')
    return np.stack([values[i] for i in range(len(rows))])


def score_dev(values, job):
    rows = job['cache']['cases']['dev']
    logits = ordered_logits(values, rows)
    predicted = (probabilities(logits) >= .5).astype(int)
    return case_diagnostics([r['gold'] for r in rows], predicted, rows), prediction(probabilities(logits), rows)


def load_task(root: Path, prepared: Path, device, workspace: Path):
    root, workspace = Path(root), Path(workspace)
    workspace.mkdir(parents=True, exist_ok=True)
    job = read_json(root / 'job.json')
    if job.get('version') != 'cs781-cjpe-v1' or source_hashes(root / 'code/cjpe') != job['runtime_source_sha256']:
        raise ValueError('Missing/changed CJPE source; use the original complete saved output')
    if source_hashes(Path(__file__).parent) != job['runtime_source_sha256']:
        raise ValueError('Load this task with its exact saved runtime, not another notebook version')
    if job['environment']['torch'] != str(torch.__version__):
        raise ValueError('Use the saved inference PyTorch build ' + job['environment']['torch'])
    if read_json(prepared / 'prepared_manifest.json')['identity'] != job['prepared_identity']:
        raise ValueError('Prepared cases differ from the learned task')
    token = root / 'tokenizer/tokenizer.json'
    if sha256_file(token) != job['parent']['tokenizer_sha256']:
        raise ValueError('Task tokenizer changed')
    cfg = replace(CJPEConfig(**job['config']), run_dir=str(workspace), prepared_dir=str(prepared))
    saved_cache = Path(job['cache']['directory'])
    cache = verify_cache(saved_cache, job['cache']['identity']) if (saved_cache / 'case_cache_manifest.json').is_file() else cache_data(prepared, token, cfg)
    if cache['manifest_sha256'] != job['cache']['manifest_sha256']:
        raise ValueError('Learned task case metadata changed')
    local = {**job, 'parent_source_dir': str(root / 'parent_source' / ('model_' + cfg.family)),
             'tokenizer_path': str(token), 'cache': cache}
    best = root / 'checkpoints/best.pt'
    receipt = read_json(root / 'checkpoints/best.pt.json')
    if sha256_file(best) != receipt['sha256'] or receipt['fingerprint'] != job['fingerprint']:
        raise ValueError('Learned task best checkpoint/receipt changed')
    payload = torch.load(best, map_location='cpu', weights_only=True, mmap=True)
    if payload['fingerprint'] != job['fingerprint']:
        raise ValueError('Checkpoint belongs to another task')
    model = build_model(local, cfg, device, load_parent=False)
    model.load_state_dict(payload['model'], strict=True)
    del payload
''')


[Markdown Cell]

Importable source part 4

In [35]:
# [Code Cell]
append_module('predict.py', r'''    return model, local, cfg, receipt


def package(values, rows, output: Path, name: str) -> Path:
    output = Path(output)
    output.mkdir(parents=True, exist_ok=True)
    validate_prediction(values, rows)
    atomic_json(output / 'prediction.json', values)
    path = output / name
    with zipfile.ZipFile(path.with_suffix('.zip.partial'), 'w', zipfile.ZIP_DEFLATED) as zipped:
        zipped.write(output / 'prediction.json', 'prediction.json')
    path.with_suffix('.zip.partial').replace(path)
    with zipfile.ZipFile(path) as zipped:
        if zipped.namelist() != ['prediction.json'] or zipped.testzip() is not None:
            raise ValueError('Submission ZIP verification failed')
    return path


# CELL: Raw 0.5 baseline, guarded calibration, frozen recipe then test predictions
def export(job_path):
    job = read_json(job_path)
    cfg = CJPEConfig(**job['config'])
    root = Path(cfg.run_dir)
    if read_json(root / 'training_summary.json')['status'] != 'complete':
        raise ValueError('Complete/resume training before exporting a completed experiment')
    best = read_json(root / 'checkpoints/best.pt.json')
    directory = root / 'predictions'
    directory.mkdir(exist_ok=True)
    artifacts = {}
    all_logits = {}
    # Only larger A/C/D models can teach B. Avoid another full train-set pass for B.
    exported_splits = ('train', 'dev') if cfg.family != 'b' else ('dev',)
    parameter_report = parallel_predict(job_path, exported_splits)
    for split in exported_splits:
        rows = job['cache']['cases'][split]
        logits = np.load(directory / (split + '_logits.npy'), allow_pickle=False)
        all_logits[split] = logits
        raw = case_diagnostics([r['gold'] for r in rows], (probabilities(logits) >= .5).astype(int), rows)
        atomic_json(directory / (split + '_raw_metrics.json'), raw)
    rows = job['cache']['cases']['dev']
    gold = [r['gold'] for r in rows]
    raw_score = metrics(gold, (probabilities(all_logits['dev']) >= .5).astype(int))['macro_f1']
    selected_epoch = next(h for h in read_json(root / 'training_summary.json')['history'] if h['epoch'] == best['epoch'])
    inherited_metric = inherited_metric_report(job, best, raw_score, selected_epoch['macro_f1'])
    if inherited_metric:
        atomic_json(root / 'inherited_checkpoint_rescoring.json', inherited_metric)
    calibration = choose_threshold(probabilities(all_logits['dev']), rows, cfg.calibrate_threshold, cfg.calibration_margin, cfg.seed)
    atomic_json(root / 'decision_rule.json', {**calibration, 'checkpoint_sha256': best['sha256'], 'prepared_identity': job['prepared_identity']})
''')


[Markdown Cell]

Importable source part 5

In [36]:
# [Code Cell]
append_module('predict.py', r'''    dev_prediction = prediction(probabilities(all_logits['dev']), rows, calibration['threshold'])
    atomic_json(directory / 'dev_prediction.json', dev_prediction)
    official(Path(job['prepared_dir']), directory / 'dev_prediction.json', calibration['selected_dev_macro_f1'], directory)
    atomic_json(directory / 'dev_metrics.json', case_diagnostics(gold, [r['pred'] for r in dev_prediction], rows))
    # The calibrated threshold is immutable before test logits are computed.
    rows = job['cache']['cases']['test']
    parallel_predict(job_path, ('test',))
    logits = np.load(directory / 'test_logits.npy', allow_pickle=False)
    final = prediction(probabilities(logits), rows, calibration['threshold'])
    path = package(final, rows, root, 'submission_' + cfg.experiment_id + '.zip')
    # Preserve the raw decision rule as a separately named valid baseline.
    raw_dir = root / 'raw_baseline'
    raw_zip = package(prediction(probabilities(logits), rows), rows, raw_dir, 'submission_' + cfg.experiment_id + '_raw.zip')
    for split in (*exported_splits, 'test'):
        for suffix in ('_logits.npy', '_order.json'):
            name = split + suffix
            artifacts[name] = sha256_file(directory / name)
    receipt = {'status': 'submission_verified', 'version': 'cjpe-output-v1', 'experiment_id': cfg.experiment_id,
               'family': cfg.family, 'method': cfg.method, 'class_order': [0, 1], 'prepared_identity': job['prepared_identity'],
               'fingerprint': job['fingerprint'], 'checkpoint_sha256': best['sha256'], 'checkpoint_epoch': best['epoch'],
               'raw_dev_macro_f1': raw_score, 'dev_macro_f1': calibration['selected_dev_macro_f1'],
               'threshold': calibration['threshold'], 'calibration': calibration, 'parameter_report': parameter_report,
               'logit_artifacts': artifacts, 'zip_path': str(path), 'zip_sha256': sha256_file(path),
               'prediction_sha256': sha256_file(root / 'prediction.json'), 'raw_zip_path': str(raw_zip),
               'inherited_checkpoint_rescoring': inherited_metric,
               'private_F1': None, 'test_labels_used': False}
    atomic_json(root / 'submission_receipt.json', receipt)
    return receipt


def inherited_metric_report(job, best, current, historical):
    """Do not compare a new batched engine to old-engine metrics as bitwise replay.

    Only the exact imported-best checksum qualifies. New optimized checkpoints
    must reproduce their own saved native dev F1 strictly. Current scores always
    come from actual reloaded tensors and current dev logits, never history.
    """
    if abs(current - historical) <= 1e-6:
        return None
    reference = job.get('engine_upgrade', {})
    if best['sha256'] != reference.get('imported_best_sha256'):
        raise ValueError('Reloaded actual optimized best weights do not reproduce saved native dev F1')
    return {'checkpoint_sha256': best['sha256'], 'stored_prior_engine_dev_macro_f1': historical,
            'current_reloaded_dev_macro_f1': current, 'source_experiment': reference['source_experiment'],
            'scope': 'Exact imported tensors rescored after an explicit checked engine transition; different batching/kernels need not give bitwise-identical threshold decisions. Original score is preserved, not substituted for the current measured score.'}


def main():
''')


[Markdown Cell]

Importable source part 6

In [37]:
# [Code Cell]
append_module('predict.py', r'''    parser = argparse.ArgumentParser()
    parser.add_argument('--job', required=True)
    args = parser.parse_args()
    print(export(args.job), flush=True)


if __name__ == '__main__':
    main()
''')


[Markdown Cell]

### preparation.py

Stream only official CJPE train/dev/test files, preserve IDs exactly and reject unexpected nonempty test labels. Audit duplicate IDs, full-text split overlaps, blank cases and binary class counts. Unknown schemas stop with diagnostics and explicit layout/field overrides, never invented labels.

[Markdown Cell]

Importable source part 1

In [38]:
# [Code Cell]
append_module('preparation.py', r'''# CELL: Bounded JSON parsing, explicit schema normalization and pinned HF inputs
import json
import os
from pathlib import Path
import re
import shutil
import sqlite3
import unicodedata
import zipfile
from .io_utils import atomic_json, identity, read_json, sha256_file

FILES = {'train': 'Phase_4_CJPE/Train/train.json', 'dev': 'Phase_4_CJPE/Train/dev.json',
         'test': 'Phase_4_CJPE/Test/test.json'}
IDS = ('id', 'case_id', 'doc_id', 'document_id')
TEXTS = ('text', 'case_text', 'document', 'content', 'paragraphs')
LABELS = ('label', 'gold', 'outcome', 'judgment', 'judgement', 'decision')
CONTAINERS = ('data', 'payload', 'cases', 'documents', 'docs', 'records', 'examples', 'items', 'dataset', 'train', 'dev', 'test')


def field(row: dict, names: tuple, required=False):
    found = [(key, row[key]) for key in names if key in row]
    if not found:
        if required:
            raise ValueError('Missing field: expected one of ' + ', '.join(names))
        return None
    if any(value != found[0][1] for _, value in found[1:]):
        raise ValueError('Contradictory redundant fields: ' + ', '.join(k for k, _ in found))
    return found[0][1]


def flatten_text(value) -> str:
    """Preserve all original text/paragraph order; never shorten a case or split labels."""
    if isinstance(value, str):
        return value
    if isinstance(value, list):
        return '\n\n'.join(flatten_text(part) for part in value)
    raise ValueError('Case text must be a string or ordered string/paragraph lists; set field overrides for a different schema')


def normalized_record(value, split: str, outer_id=None, fields=None) -> dict:
    fields = fields or {}
    if isinstance(value, str) and outer_id is not None:
        value = {'id': outer_id, 'text': value}
    if not isinstance(value, dict):
        raise ValueError('Each case must be an object containing its original ID and text')
    row = dict(value)
    get = lambda name, aliases, required=False: row[fields[name]] if name in fields else field(row, aliases, required)
    case_id = get('id', IDS)
''')


[Markdown Cell]

Importable source part 2

In [39]:
# [Code Cell]
append_module('preparation.py', r'''    if case_id is None:
        case_id = outer_id
    elif outer_id is not None and case_id != outer_id:
        raise ValueError('Outer case key and record ID disagree')
    if not isinstance(case_id, str) or not case_id.strip():
        raise ValueError('Original case IDs must be non-empty strings; no .txt removal or invented IDs')
    text = flatten_text(get('text', TEXTS, True))
    if len(text) > 50_000_000:
        raise ValueError(f'{case_id}: case exceeds 50M characters; inspect schema before encoding (no text was truncated)')
    label = get('label', LABELS)
    if split == 'test':
        if label is not None and label != []:
            raise ValueError('Nonempty test annotation found; use only official unlabelled test input')
        return {'id': case_id, 'text': text}
    if isinstance(label, str) and label in ('0', '1'):
        label = int(label)
    if type(label) is not int or label not in (0, 1):
        raise ValueError(f'{case_id}: train/dev needs a binary integer label 0/1; booleans/unknown labels rejected')
    return {'id': case_id, 'text': text, 'gold': label}


def layout(path: Path, prefix='') -> tuple[str, str, dict]:
    """Inspect immediate fields with bounded metadata; named wrappers may be nested."""
    import ijson
    kind = None
    fields = {}
    with Path(path).open('rb') as stream:
        for current, event, value in ijson.parse(stream):
            if current == prefix and event in ('start_array', 'start_map') and kind is None:
                kind = event
                if kind == 'start_array':
                    return 'items', (prefix + '.item').lstrip('.'), {'kind': 'array', 'prefix': prefix}
            if current == prefix and event == 'map_key' and (len(fields) < 32 or value in CONTAINERS):
                fields[value] = None
            for key in tuple(fields):
                if fields[key] is None and current == (prefix + '.' + key).lstrip('.') and event not in ('map_key', 'end_map', 'end_array'):
                    fields[key] = event
    if kind != 'start_map':
        raise ValueError('Expected JSON array or object at prefix ' + prefix)
    containers = [key for key in CONTAINERS if fields.get(key) in ('start_array', 'start_map')]
    if any(fields.get(key) in ('string', 'number') for key in IDS) and any(key in fields for key in TEXTS):
        return 'single', prefix, {'kind': 'record', 'fields': fields, 'prefix': prefix}
    if len(containers) > 1:
        raise ValueError('Multiple document containers; set SOURCE_LAYOUTS explicitly: ' + repr(containers))
    if containers:
        return layout(path, (prefix + '.' + containers[0]).lstrip('.'))
    if any(key in fields for key in TEXTS):
        raise ValueError('Column-oriented/unknown JSON detected. Set SOURCE_LAYOUTS/FIELD_MAP explicitly; use record arrays or ID-record maps.')
''')


[Markdown Cell]

Importable source part 3

In [40]:
# [Code Cell]
append_module('preparation.py', r'''    return 'mapping', prefix, {'kind': 'ID_mapping', 'fields': fields, 'prefix': prefix}


def stream_cases(path: Path, split: str, specification=None, fields=None):
    """At most one case object/text in memory; supported layouts never load the full array."""
    import ijson
    specification = specification or {}
    if specification:
        mode, prefix = specification['mode'], specification.get('prefix', '')
    else:
        mode, prefix, _ = layout(path)
    if mode not in ('items', 'mapping', 'single'):
        raise ValueError('SOURCE_LAYOUTS mode must be items, mapping or single')
    with Path(path).open('rb') as stream:
        if mode == 'mapping':
            for key, value in ijson.kvitems(stream, prefix):
                yield normalized_record(value, split, key, fields)
        else:
            for value in ijson.items(stream, prefix):
                yield normalized_record(value, split, fields=fields)


def download_official(repository, revision, destination, token, path_map=None):
    from huggingface_hub import HfApi, hf_hub_download
    api = HfApi(token=token)
    try:
        commit = api.dataset_info(repository, revision=revision).sha
        names = set(api.list_repo_files(repository, repo_type='dataset', revision=commit))
    except Exception as error:
        raise RuntimeError('Enable Kaggle Internet and HF_TOKEN/HF Secret; accept the official HF dataset access terms') from error
    selected = {s: (path_map or {}).get(s, name) for s, name in FILES.items()}
    if not all(name in names for name in selected.values()):
        raise ValueError('CJPE paths changed; set HF_PATH_MAP using the repository listing. Do not select another phase\'s train.json.')
    paths = {split: Path(hf_hub_download(repo_id=repository, repo_type='dataset', revision=commit,
             filename=name, token=token, local_dir=str(destination))) for split, name in selected.items()}
    return paths, {'repository': repository, 'commit': commit, 'paths': selected}


# CELL: Case-level duplicate audit and canonical bounded JSONL preparation
def prepare(paths: dict, output: Path, provenance: dict, evaluator: Path, layouts=None, field_map=None) -> dict:
    from tqdm.auto import tqdm
    output = Path(output)
    output.mkdir(parents=True, exist_ok=True)
    choices = {'provenance': provenance, 'source_sha256': {k: sha256_file(v) for k, v in paths.items()},
               'layouts': layouts or {}, 'field_map': field_map or {}}
    existing = output / 'prepared_manifest.json'
    if existing.is_file():
        saved = read_json(existing)
''')


[Markdown Cell]

Importable source part 4

In [41]:
# [Code Cell]
append_module('preparation.py', r'''        if saved['source_identity'] != identity(choices):
            raise ValueError('Prepared sources/schema changed; use a new OUTPUT_ROOT')
        verify_prepared(output)
        return saved
    structures = {}
    for split, path in paths.items():
        try:
            if layouts and split in layouts:
                structures[split] = {'explicit': layouts[split]}
            else:
                mode, prefix, report = layout(path)
                structures[split] = {**report, 'mode': mode, 'prefix': prefix}
        except (ValueError, KeyError) as error:
            structures[split] = {'error': str(error), 'file_bytes': Path(path).stat().st_size}
            atomic_json(output.parent / 'input_structure_report.json', structures)
            raise
        atomic_json(output.parent / 'input_structure_report.json', structures)
    db = sqlite3.connect(output / 'audit.sqlite')
    db.execute('CREATE TABLE IF NOT EXISTS cases(split TEXT,id TEXT,sha TEXT,norm TEXT,chars INTEGER,gold INTEGER,PRIMARY KEY(split,id))')
    db.execute('DELETE FROM cases')
    counts = {}
    artifacts = {}
    try:
        for split, path in paths.items():
            count, blanks, chars, labels = 0, 0, 0, [0, 0]
            destination = output / (split + '.jsonl')
            temporary = destination.with_suffix('.jsonl.partial')
            with temporary.open('w', encoding='utf-8', newline='\n') as target:
                for row in tqdm(stream_cases(path, split, (layouts or {}).get(split), (field_map or {}).get(split)), desc='Audit ' + split, unit='cases'):
                    text = row['text']
                    normalized = re.sub(r'\s+', ' ', unicodedata.normalize('NFKC', text)).strip()
                    try:
                        db.execute('INSERT INTO cases VALUES(?,?,?,?,?,?)', (split, row['id'], identity(text), identity(normalized), len(text), row.get('gold')))
                    except sqlite3.IntegrityError as error:
                        raise ValueError(f'{split}: duplicate original case ID {row["id"]}; no silent ID rewriting/deduplication') from error
                    target.write(json.dumps(row, ensure_ascii=False, allow_nan=False) + '\n')
                    count += 1
                    chars += len(text)
                    blanks += not bool(text.strip())
                    if split != 'test':
                        labels[row['gold']] += 1
                    if count % 1000 == 0:
                        db.commit()
                target.flush()
                os.fsync(target.fileno())
            if not count:
                raise ValueError(split + ': no cases were found; inspect input_structure_report.json')
            os.replace(temporary, destination)
''')


[Markdown Cell]

Importable source part 5

In [42]:
# [Code Cell]
append_module('preparation.py', r'''            db.commit()
            counts[split] = {'cases': count, 'characters': chars, 'blank_cases': blanks,
                             'class_counts': labels if split != 'test' else None}
            artifacts[destination.name] = {'sha256': sha256_file(destination), 'bytes': destination.stat().st_size}
        overlaps = list(db.execute('SELECT a.split,a.id,b.split,b.id FROM cases a JOIN cases b ON a.norm=b.norm AND a.split<b.split WHERE a.chars>=100 AND b.chars>=100 LIMIT 50'))
        duplicate_groups = db.execute('SELECT COUNT(*) FROM (SELECT norm FROM cases WHERE split="train" GROUP BY norm HAVING COUNT(*)>1)').fetchone()[0]
        conflicting_groups = db.execute('SELECT COUNT(*) FROM (SELECT norm FROM cases WHERE split="train" GROUP BY norm HAVING COUNT(DISTINCT gold)>1)').fetchone()[0]
        audit = {'counts': counts, 'train_duplicate_text_groups': duplicate_groups,
                 'train_conflicting_label_text_groups': conflicting_groups,
                 'cross_split_exact_or_whitespace_Unicode_equivalent_examples': overlaps,
                 'near_duplicate_scope': 'Only exact and whitespace/Unicode-equivalent full texts are compared; semantic/approximate near-duplicate audit is not claimed',
                 'training_cases_dropped': 0, 'test_gold_used': False}
        atomic_json(output / 'data_audit.json', audit)
        if overlaps:
            raise ValueError('Full-case text overlaps official splits; see data_audit.json. Training is blocked rather than silently leaking or dropping cases.')
        if not all(counts['train']['class_counts']):
            raise ValueError('Both outcome classes must occur in training; inspect class counts/label schema')
        shutil.copyfile(evaluator, output / 'evaluate_phase4.py')
        artifacts.update({name: {'sha256': sha256_file(output / name), 'bytes': (output / name).stat().st_size}
                          for name in ('data_audit.json', 'evaluate_phase4.py')})
        manifest = {'version': 'cjpe-data-v1', 'status': 'complete', 'source_identity': identity(choices),
                    'source': choices, 'counts': counts, 'artifacts': artifacts, 'test_gold_used': False}
        manifest['identity'] = identity(manifest)
        atomic_json(existing, manifest)
    finally:
        db.close()
    archive = output.parent / 'phase4_prepared.zip'
    with zipfile.ZipFile(archive.with_suffix('.zip.partial'), 'w', zipfile.ZIP_DEFLATED, compresslevel=1) as zipped:
        for name in [*artifacts, 'prepared_manifest.json']:
            zipped.write(output / name, name)
    archive.with_suffix('.zip.partial').replace(archive)
    return manifest


def verify_prepared(root: Path) -> dict:
    manifest = read_json(root / 'prepared_manifest.json')
    if (manifest.get('version') != 'cjpe-data-v1' or manifest.get('status') != 'complete'
            or identity({k: v for k, v in manifest.items() if k != 'identity'}) != manifest['identity']):
        raise ValueError('Invalid CJPE prepared manifest; Phase 2/3 outputs are different tasks')
    for name, receipt in manifest['artifacts'].items():
        if Path(name).name != name or '\\' in name or (root / name).stat().st_size != receipt['bytes'] or sha256_file(root / name) != receipt['sha256']:
            raise ValueError('Missing/changed prepared artifact: ' + name)
    return manifest


def prepared_input(cfg) -> Path:
    explicit = Path(cfg.prepared_dir) if cfg.prepared_dir else None
    if explicit and not explicit.exists():
''')


[Markdown Cell]

Importable source part 6

In [43]:
# [Code Cell]
append_module('preparation.py', r'''        raise FileNotFoundError('Explicit prepared_dir does not exist: ' + str(explicit))
    candidates = ([explicit] if explicit.is_dir() else []) if explicit else [p.parent for p in Path(cfg.input_root).rglob('prepared_manifest.json')
        if read_json(p).get('version') == 'cjpe-data-v1']
    if not candidates:
        archives = [explicit] if explicit else list(Path(cfg.input_root).rglob('phase4_prepared.zip'))
        for path in archives:
            if path is None or not path.is_file() or path.suffix != '.zip':
                continue
            root = Path(cfg.run_dir) / 'prepared_input'
            root.mkdir(parents=True, exist_ok=True)
            with zipfile.ZipFile(path) as zipped:
                manifest = json.loads(zipped.read('prepared_manifest.json'))
                names = [*manifest['artifacts'], 'prepared_manifest.json']
                if len(zipped.namelist()) != len(set(zipped.namelist())) or any(Path(n).name != n or '\\' in n for n in names):
                    raise ValueError('Unsafe/duplicate prepared ZIP members')
                for name in names:
                    with zipped.open(name) as source, (root / name).open('wb') as target:
                        shutil.copyfileobj(source, target)
            candidates.append(root)
    if not candidates:
        raise FileNotFoundError('Attach completed Phase 4 notebook 00 output or its phase4_prepared.zip')
    manifests = [verify_prepared(root) for root in candidates]
    if len({value['identity'] for value in manifests}) != 1:
        raise ValueError('Different CJPE prepared versions attached; set prepared_dir to the exact folder or ZIP')
    return candidates[0]
''')


[Markdown Cell]

### prepare.py

Verify complete own Phase 1 checkpoints, actual original architecture sources, matching tokenizer/metadata and data identity. A saved CJPE task can resume directly from its full recovery state. Each KD teacher is validated against its own saved runtime and aligned train/dev logits; unrelated student-parser edits are not compared to teacher bytes.

[Markdown Cell]

Importable source part 1

In [44]:
# [Code Cell]
append_module('prepare.py', r'''# CELL: Own-foundation verification and exact task recovery identities
from dataclasses import asdict, replace
from pathlib import Path
import shutil
import numpy as np
import torch
from . import VERSION
from .config import CJPEConfig
from .data import cache_data, CaseDataset, collate, training_weights
from .io_utils import atomic_json, identity, read_json, sha256_file
from .parents import parent_class, source_hashes, tokenizer_inputs, verify_parent
from .preparation import prepared_input
from .state import Checkpoints


def require_hardware():
    if not torch.cuda.is_available() or torch.cuda.device_count() != 2 or not torch.distributed.is_nccl_available():
        raise RuntimeError('Select Kaggle T4 x2: two CUDA devices and NCCL are required')
    for rank in range(2):
        name = torch.cuda.get_device_name(rank)
        if 'T4' not in name:
            raise RuntimeError('Expected T4 x2; found ' + name)
        print('GPU', rank, ':', name, flush=True)


def recovery_candidates(cfg):
    local = Path(cfg.run_dir) / 'job.json'
    if cfg.upgrade_run_dir and not local.is_file():
        from .upgrade import upgrade_candidate
        return [upgrade_candidate(cfg)]
    paths = [local] if local.is_file() else ([Path(cfg.resume_run_dir) / 'job.json'] if cfg.resume_run_dir else sorted(Path(cfg.input_root).rglob('job.json')))
    result = []
    for path in paths:
        value = read_json(path)
        if value.get('config', {}).get('experiment_id') != cfg.experiment_id:
            if cfg.resume_run_dir:
                raise ValueError('Explicit resume root belongs to another experiment')
            continue
        if value.get('version') != VERSION or value['runtime_source_sha256'] != source_hashes(Path(__file__).parent):
            raise ValueError('Exact student recovery needs its original complete CJPE notebook/runtime')
        if CJPEConfig(**value['config']).choices() != cfg.choices():
            raise ValueError('Learning/head/chunk choices changed: use a new experiment ID, not an old optimizer')
        if value['environment']['torch'] != str(torch.__version__):
            raise ValueError('Exact recovery requires saved PyTorch build ' + value['environment']['torch'])
        if source_hashes(path.parent / 'code/cjpe') != value['runtime_source_sha256']:
            raise ValueError('Saved student source files changed or are incomplete')
        result.append((path.parent, value))
    if len({value['fingerprint'] for _, value in result}) > 1:
''')


[Markdown Cell]

Importable source part 2

In [45]:
# [Code Cell]
append_module('prepare.py', r'''        raise ValueError('Conflicting versions share this ID; set resume_run_dir to one exact task root')
    if result and not (Path(cfg.run_dir) / 'checkpoints').exists() and not any((root / 'checkpoints').is_dir() for root, _ in result):
        raise FileNotFoundError('The saved job has no actual checkpoint folder. Attach the full output, not observations.zip.')
    return result


# CELL: Completed larger own CJPE teachers, each checked against its own runtime
def teacher_inputs(job, cfg, catalog):
    destination = Path(cfg.run_dir) / 'teacher_cache.npy'
    reference_path = Path(cfg.run_dir) / 'teacher_reference.json'
    if destination.is_file() and reference_path.is_file():
        reference = read_json(reference_path)
        if sha256_file(destination) != reference['cache_sha256'] or reference['prepared_identity'] != job['prepared_identity']:
            raise ValueError('Saved teacher cache/data identity changed')
        return reference
    paths = [Path(cfg.teacher_run_dir) / 'job.json'] if cfg.teacher_run_dir else sorted(Path(cfg.input_root).rglob('job.json'))
    diagnostics, eligible = [], []
    expected_order = [{'id': r['id']} for r in job['cache']['cases']['train']]
    expected_dev = job['cache']['cases']['dev']
    from .metrics import metrics, probabilities
    for path in paths:
        row = {'task_root': str(path.parent), 'accepted': False, 'reasons': []}
        try:
            teacher = read_json(path)
            family = teacher.get('config', {}).get('family')
            if teacher.get('version') != VERSION or family not in ('a', 'c', 'd'):
                if cfg.teacher_run_dir:
                    raise ValueError('Explicit teacher is not a larger Phase 4 A/C/D task')
                continue
            root = path.parent
            receipt = read_json(root / 'submission_receipt.json')
            if read_json(root / 'training_summary.json')['status'] != 'complete' or receipt['status'] != 'submission_verified':
                raise ValueError('Teacher is paused or has not completed export')
            if receipt['prepared_identity'] != job['prepared_identity'] or receipt.get('class_order') != [0, 1]:
                raise ValueError('Teacher data/outcome order differs')
            if receipt['fingerprint'] != teacher['fingerprint'] or teacher['prepared_identity'] != job['prepared_identity']:
                raise ValueError('Teacher job and submission receipt belong to different runs/data')
            if receipt['parameter_report']['foundation_unique_parameters'] <= job['parent_parameter_count']:
                raise ValueError('Teacher foundation must be larger than B for this KD experiment')
            if source_hashes(root / 'code/cjpe') != teacher['runtime_source_sha256']:
                raise ValueError('Teacher code differs from its OWN saved source manifest')
            if (teacher['parent_contract']['source_sha256'] != catalog[family]['source_sha256']
                    or source_hashes(root / 'parent_source' / ('model_' + family)) != catalog[family]['source_sha256']):
                raise ValueError('Teacher does not use an approved own Phase 1 architecture')
            best = root / 'checkpoints/best.pt'
            checkpoint_receipt = read_json(root / 'checkpoints/best.pt.json')
            if (sha256_file(best) != receipt['checkpoint_sha256'] or checkpoint_receipt['sha256'] != receipt['checkpoint_sha256']
                    or checkpoint_receipt['fingerprint'] != teacher['fingerprint']):
''')


[Markdown Cell]

Importable source part 3

In [46]:
# [Code Cell]
append_module('prepare.py', r'''                raise ValueError('Teacher checkpoint bytes and receipt disagree')
            for split, expected in (('train', expected_order), ('dev', [{'id': r['id']} for r in expected_dev])):
                for suffix in ('_logits.npy', '_order.json'):
                    name = split + suffix
                    if sha256_file(root / 'predictions' / name) != receipt['logit_artifacts'][name]:
                        raise ValueError('Teacher exported logit/order checksum changed: ' + name)
                if read_json(root / 'predictions' / (split + '_order.json')) != expected:
                    raise ValueError('Teacher logits are not aligned to original case IDs')
                logits = np.load(root / 'predictions' / (split + '_logits.npy'), mmap_mode='r', allow_pickle=False)
                if logits.shape != (len(expected), 2) or not np.isfinite(logits).all():
                    raise ValueError('Teacher logits are nonfinite or have wrong case/class dimensions')
            dev_logits = np.load(root / 'predictions/dev_logits.npy', allow_pickle=False)
            score = metrics([r['gold'] for r in expected_dev], (probabilities(dev_logits) >= .5).astype(int))['macro_f1']
            if abs(score - receipt['raw_dev_macro_f1']) > 1e-9:
                raise ValueError('Teacher exported dev logits do not reproduce its native F1')
            row['accepted'] = True
            row['raw_dev_macro_f1'] = score
            eligible.append((score, root, teacher, receipt))
        except (ValueError, KeyError, OSError) as error:
            row['reasons'].append(str(error))
        diagnostics.append(row)
    atomic_json(Path(cfg.run_dir) / 'teacher_input_report.json', diagnostics)
    print({'teacher_candidates': diagnostics}, flush=True)
    if not eligible:
        raise FileNotFoundError('No compatible COMPLETE larger PHASE 4 teacher. Attach full 01/02/04/05 CJPE output with actual best.pt/receipt, source and train/dev logits; see teacher_input_report.json. Phase 3 models and observation ZIPs are not CJPE teachers.')
    _, root, teacher, receipt = max(eligible, key=lambda x: (x[0], str(x[1])))
    values = np.load(root / 'predictions/train_logits.npy', allow_pickle=False)
    with destination.with_suffix('.npy.partial').open('wb') as stream:
        np.save(stream, values)
    destination.with_suffix('.npy.partial').replace(destination)
    reference = {'teacher_experiment': teacher['config']['experiment_id'], 'teacher_checkpoint_sha256': receipt['checkpoint_sha256'],
                 'teacher_fingerprint': teacher['fingerprint'], 'teacher_source_sha256': teacher['runtime_source_sha256'],
                 'teacher_raw_dev_macro_f1': receipt['raw_dev_macro_f1'], 'prepared_identity': job['prepared_identity'],
                 'class_order': [0, 1], 'cache_sha256': sha256_file(destination), 'training_only': True,
                 'teacher_foundation_parameters': receipt['parameter_report']['foundation_unique_parameters'],
                 'student_foundation_parameters': job['parent_parameter_count'], 'test_labels_used': False}
    atomic_json(reference_path, reference)
    return reference


# CELL: Portable full-state resumption and explicit total-epoch extension
def prepare_job(cfg, catalog):
    cfg.validate()
    run = Path(cfg.run_dir)
    run.mkdir(parents=True, exist_ok=True)
    prepared = prepared_input(cfg)
    manifest = read_json(prepared / 'prepared_manifest.json')
    candidates = recovery_candidates(cfg)
''')


[Markdown Cell]

Importable source part 4

In [47]:
# [Code Cell]
append_module('prepare.py', r'''    prior = candidates[0] if candidates else None
    source = run / 'parent_source' / ('model_' + cfg.family)
    source.mkdir(parents=True, exist_ok=True)
    small = run / 'tokenizer'
    small.mkdir(exist_ok=True)
    contract = catalog[cfg.family]
    if prior:
        root, previous = prior
        parent = previous['parent']
        for name in contract['source_sha256']:
            origin = root / 'parent_source' / ('model_' + cfg.family) / name
            if sha256_file(origin) != contract['source_sha256'][name]:
                raise ValueError('Saved foundation source changed')
            if origin.resolve() != (source / name).resolve():
                shutil.copyfile(origin, source / name)
        for name, key in (('tokenizer.json', 'tokenizer_sha256'), ('metadata.json', 'metadata_sha256')):
            origin = root / 'tokenizer' / name
            if sha256_file(origin) != parent[key]:
                raise ValueError('Saved tokenizer snapshot changed')
            if origin.resolve() != (small / name).resolve():
                shutil.copyfile(origin, small / name)
        for name in ('teacher_cache.npy', 'teacher_reference.json'):
            origin = root / name
            if origin.is_file() and origin.resolve() != (run / name).resolve():
                shutil.copyfile(origin, run / name)
    else:
        root, parent = verify_parent(cfg, contract)
        token, meta, hashes = tokenizer_inputs(cfg, contract)
        parent.update(hashes)
        for name in contract['source_sha256']:
            shutil.copyfile(root / 'code' / ('model_' + cfg.family) / name, source / name)
        shutil.copyfile(token, small / 'tokenizer.json')
        shutil.copyfile(meta, small / 'metadata.json')
    if cfg.expected_parent_sha256 and parent['checkpoint_sha256'] != cfg.expected_parent_sha256:
        raise ValueError('Different selected Phase 1 checkpoint; use the audited one or explicitly accept another verified OWN continuation with expected_parent_sha256=None')
    cache = cache_data(prepared, small / 'tokenizer.json', cfg)
    Config, Model = parent_class(str(source), cfg.family, contract)
    with torch.device('meta'):
        probe = Model(Config(**parent['config']))
    parameters = sum(p.numel() for p in probe.parameters())
    assert 50_000_000 < parameters < 300_000_000
    del probe
    environment = {'torch': str(torch.__version__), 'numpy': np.__version__, 'tokenizers': __import__('tokenizers').__version__}
    runtime = source_hashes(Path(__file__).parent)
    job = {'version': VERSION, 'config': asdict(cfg), 'prepared_dir': str(prepared), 'prepared_identity': manifest['identity'],
           'parent': parent, 'parent_contract': contract, 'parent_source_dir': str(source),
           'parent_parameter_count': parameters, 'tokenizer_path': str(small / 'tokenizer.json'),
           'cache': cache, 'runtime_source_sha256': runtime, 'environment': environment,
''')


[Markdown Cell]

Importable source part 5

In [48]:
# [Code Cell]
append_module('prepare.py', r'''           'resume_roots': [str(root) for root, _ in candidates], 'class_order': [0, 1]}
    if prior and prior[1].get('engine_upgrade'):
        job['engine_upgrade'] = prior[1]['engine_upgrade']
        atomic_json(run / 'engine_upgrade_report.json', job['engine_upgrade'])
    if cfg.method == 'kd':
        job['teacher'] = teacher_inputs(job, cfg, catalog)
    fingerprint = identity({'version': VERSION, 'choices': cfg.choices(), 'prepared': manifest['identity'],
                            'parent': parent['checkpoint_sha256'], 'tokenizer': parent['tokenizer_sha256'],
                            'runtime': runtime, 'environment': environment, 'cache': cache['identity'],
                            'case_metadata': cache['manifest_sha256'], 'teacher': job.get('teacher')})
    upgrading = bool(prior and prior[1].get('_engine_upgrade'))
    if prior and not upgrading and prior[1]['fingerprint'] != fingerprint:
        raise ValueError('Exact student recovery fingerprint changed')
    job['fingerprint'] = fingerprint
    if upgrading:
        from .upgrade import import_upgrade
        job['engine_upgrade'] = import_upgrade(job, prior, cfg)
        job['resume_roots'] = []
    manager = Checkpoints(run, fingerprint, cfg.min_free_bytes, job['resume_roots'])
    latest = manager.latest()
    effective, extending = cfg.max_epochs, False
    if latest:
        payload = manager.load(latest)
        original = payload['config']['max_epochs']
        effective = max(original, cfg.max_epochs)
        if cfg.target_total_epochs is not None:
            if cfg.target_total_epochs < payload['cursor']['epoch']:
                raise ValueError('Target total cannot be less than already completed epochs')
            effective = cfg.target_total_epochs
            extending = (cfg.target_total_epochs > payload['cursor']['epoch'] and cfg.target_total_epochs != payload['config'].get('target_total_epochs'))
        if payload['cursor']['complete'] and cfg.target_total_epochs is None and cfg.max_epochs > original:
            raise ValueError('Use TARGET_TOTAL_EPOCHS to extend a completed task; do not change only MAX_EPOCHS')
        schedule = prior[1].get('schedule_epochs', original)
        del payload
    else:
        if prior and prior[1].get('resume_checkpoint'):
            raise ValueError('Saved recovery checkpoint is missing; no silent fresh restart')
        schedule = cfg.max_epochs
        if cfg.target_total_epochs is not None:
            effective = cfg.target_total_epochs
    cfg = replace(cfg, max_epochs=effective)
    job.update(config=asdict(cfg), extension_requested=extending, schedule_epochs=schedule, resume_checkpoint=str(latest[0]) if latest else None)
    atomic_json(run / 'job.json', job)
    atomic_json(run / 'task_config.json', asdict(cfg))
    return job


def sanity(job, cfg):
''')


[Markdown Cell]

Importable source part 6

In [49]:
# [Code Cell]
append_module('prepare.py', r'''    from .model import build_model, forward
    device = torch.device('cuda', 0)
    model = build_model(job, cfg, device, load_parent=not bool(job['resume_checkpoint']))
    if job['resume_checkpoint']:
        payload = torch.load(job['resume_checkpoint'], map_location='cpu', weights_only=True, mmap=True)
        model.load_state_dict(payload['model'], strict=True)
        del payload
    dataset = CaseDataset(job['cache'], 'train', cfg)
    index = min(range(len(dataset)), key=lambda i: dataset.rows[i]['length'])
    batch = collate([dataset[index], dataset[-1]])
    model.train()
    with torch.autocast('cuda', dtype=torch.float16):
        logits = forward(model, batch, device)
        loss = torch.nn.functional.cross_entropy(logits.float(), batch['targets'].to(device), ignore_index=-100)
    if logits.shape != (2, 2) or not torch.isfinite(logits).all() or not torch.isfinite(loss):
        raise ValueError('Full-size CJPE model failed finite shape check')
    loss.backward()
    gradients = [p.grad for p in model.backbone.parameters() if p.requires_grad]
    if not gradients or not all(g is not None and torch.isfinite(g).all() for g in gradients):
        raise ValueError('Active backbone adaptation has missing/nonfinite gradients')
    if model.classifier.weight.grad is None or not torch.isfinite(model.classifier.weight.grad).all():
        raise ValueError('Classification head has missing/nonfinite gradients')
    weights = training_weights(job['cache'], cfg)
    result = {**model.parameter_report, 'logit_shape': list(logits.shape), 'finite_shape_and_gradients': True,
              'training_cases': len(dataset), 'training_class_weights': weights.tolist(),
              'chunk_counts_quantiles': np.quantile([r['chunks'] for r in dataset.rows], [0, .5, .9, .99, 1]).tolist(),
              'scope': 'Actual full architecture/starting tensors sanity; no model quality claim'}
    atomic_json(Path(cfg.run_dir) / 'sanity_report.json', result)
    dataset.close()
    del model, logits, loss
    torch.cuda.empty_cache()
    return result
''')


[Markdown Cell]

### profile.py

Measure discarded-weight full-architecture timings at length quantiles on two T4s. Report actual all-case work and a rough full-epoch range. This probe is not training on a sampled dataset, does not save learned weights and makes no F1 claim.

[Markdown Cell]

Importable source part 1

In [50]:
# [Code Cell]
append_module('profile.py', r'''# CELL: Discarded-weight performance probe; all-case epoch work is estimated explicitly
import math
from pathlib import Path
import socket
import time
import numpy as np
import torch
import torch.distributed as dist
import torch.multiprocessing as mp
from torch.nn.parallel import DistributedDataParallel as DDP
from .config import CJPEConfig
from .data import CaseDataset, collate, training_weights
from .io_utils import atomic_json, read_json
from .model import build_model, forward
from .state import seed_everything
from .train import objective, optimizer_for


def profile_worker(rank, job, port):
    cfg = CJPEConfig(**job['config'])
    torch.set_num_threads(2)
    torch.cuda.set_device(rank)
    device = torch.device('cuda', rank)
    dist.init_process_group('nccl', init_method=f'tcp://127.0.0.1:{port}', rank=rank, world_size=2)
    try:
        seed_everything(cfg.seed)
        model = build_model(job, cfg, device, load_parent=not bool(job['resume_checkpoint']))
        if job['resume_checkpoint']:
            model.load_state_dict(torch.load(job['resume_checkpoint'], map_location='cpu', weights_only=True, mmap=True)['model'])
        model = DDP(model, device_ids=[rank], broadcast_buffers=False)
        optimizer = optimizer_for(model, cfg)
        weights = torch.tensor(training_weights(job['cache'], cfg), device=device)
        dataset = CaseDataset(job['cache'], 'train', cfg)
        ordered = sorted(range(len(dataset)), key=lambda i: dataset.rows[i]['chunks'])
        # A performance probe only: original foundations/task checkpoints are never changed.
        selected = [ordered[round((len(ordered) - 1) * q)] for q in (.25, .5, .75, .9, .99)]
        measurements = []
        scaler = torch.amp.GradScaler('cuda', init_scale=4096)
        for index in selected:
            batch = collate([dataset[index]])
            optimizer.zero_grad(set_to_none=True)
            torch.cuda.synchronize()
            dist.barrier()
            torch.cuda.reset_peak_memory_stats()
            started = time.monotonic()
            with torch.autocast('cuda', dtype=torch.float16):
                logits = forward(model, batch, device)
                loss = objective(logits, batch['targets'].to(device), weights, batch['teacher'].to(device), cfg)
''')


[Markdown Cell]

Importable source part 2

In [51]:
# [Code Cell]
append_module('profile.py', r'''            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            torch.cuda.synchronize()
            stats = torch.tensor([time.monotonic() - started, torch.cuda.max_memory_allocated() / 2**30], device=device)
            dist.all_reduce(stats, op=dist.ReduceOp.MAX)
            measurements.append({'case_chunks': dataset.rows[index]['chunks'], 'seconds_per_case_with_update': float(stats[0]), 'peak_GiB_per_GPU': float(stats[1])})
        if rank == 0:
            chunks = sum(row['chunks'] for row in dataset.rows)
            ratios = [row['seconds_per_case_with_update'] / row['case_chunks'] for row in measurements[1:]]
            estimate = chunks * float(np.median(ratios)) / 2
            atomic_json(Path(cfg.run_dir) / 'profile_report.json', {
                'status': 'performance_probe_complete', 'training_cases': len(dataset), 'all_training_chunks': chunks,
                'measurements': measurements, 'rough_training_epoch_seconds': estimate,
                'rough_epoch_range_seconds': [estimate * .65, estimate * 2],
                'scope': 'Discarded-weight timings on length quantiles, not reduced-data training or F1. Excludes full dev/export/checkpoint time and worst-case length outliers. Measure the first real epoch before allocating all six runs.',
                'model_parameter_report': model.module.parameter_report})
        dataset.close()
    finally:
        dist.destroy_process_group()


def run_profile(job_path):
    job = read_json(job_path)
    if job['config']['method'] == 'kd':
        raise ValueError('Profile supervised B first; KD needs its completed larger teacher')
    with socket.socket() as listener:
        listener.bind(('127.0.0.1', 0))
        port = listener.getsockname()[1]
    mp.spawn(profile_worker, args=(job, port), nprocs=2, join=True)
    return read_json(Path(job['config']['run_dir']) / 'profile_report.json')
''')


[Markdown Cell]

### select.py

CPU-only comparison uses checksummed aligned dev logits. Default selects a single learned model. Equal-weight pairs are optional only if official rules permit and a conditional grouped improvement gate passes. Dev selection is not an independent test estimate.

[Markdown Cell]

Importable source part 1

In [52]:
# [Code Cell]
append_module('select.py', r'''# CELL: Receipt-verified development comparison; no test labels or leaderboard selection
import argparse
from itertools import combinations
from pathlib import Path
import re
import unicodedata
import numpy as np
from sklearn.model_selection import StratifiedGroupKFold
from .io_utils import atomic_json, identity, read_json, records, sha256_file
from .metrics import choose_threshold, metrics, official, prediction, probabilities
from .predict import package
from .preparation import verify_prepared


def completed_outputs(input_root, prepared, explicit=None):
    manifest = verify_prepared(Path(prepared))
    paths = [Path(root) / 'submission_receipt.json' for root in explicit] if explicit else sorted(Path(input_root).rglob('submission_receipt.json'))
    accepted, diagnostics, seen = [], [], set()
    dev_rows = list(records(Path(prepared) / 'dev.jsonl'))
    for row in dev_rows:
        row['group'] = identity(re.sub(r'\s+', ' ', unicodedata.normalize('NFKC', row['text'])).strip())
    expected = [{'id': r['id']} for r in dev_rows]
    for path in paths:
        row = {'root': str(path.parent), 'accepted': False}
        try:
            receipt = read_json(path)
            if receipt.get('version') != 'cjpe-output-v1':
                continue
            root = path.parent
            if receipt['status'] != 'submission_verified' or read_json(root / 'training_summary.json')['status'] != 'complete':
                raise ValueError('Output is not a completed learned/exported task')
            if receipt['prepared_identity'] != manifest['identity'] or receipt['class_order'] != [0, 1]:
                raise ValueError('Case-data or binary class order differs')
            if receipt['fingerprint'] in seen:
                continue
            # CPU comparison needs receipts and numeric predictions, not GB-sized model tensors.
            for name in ('dev_logits.npy', 'dev_order.json'):
                if sha256_file(root / 'predictions' / name) != receipt['logit_artifacts'][name]:
                    raise ValueError('Changed prediction artifact: ' + name)
            if read_json(root / 'predictions/dev_order.json') != expected:
                raise ValueError('Development logits have missing/reordered original IDs')
            probability = probabilities(np.load(root / 'predictions/dev_logits.npy', allow_pickle=False))
            score = metrics([r['gold'] for r in dev_rows], probability >= .5)['macro_f1']
            calibrated = metrics([r['gold'] for r in dev_rows], probability >= receipt['threshold'])['macro_f1']
            if abs(score - receipt['raw_dev_macro_f1']) > 1e-9 or abs(calibrated - receipt['dev_macro_f1']) > 1e-9:
                raise ValueError('Receipt F1 is not reproduced by actual logits')
            accepted.append({'root': root, 'receipt': receipt, 'dev_probability': probability})
            seen.add(receipt['fingerprint'])
''')


[Markdown Cell]

Importable source part 2

In [53]:
# [Code Cell]
append_module('select.py', r'''            row.update(accepted=True, raw_dev_macro_f1=score, calibrated_dev_macro_f1=calibrated)
        except (ValueError, KeyError, OSError) as error:
            row['error'] = str(error)
        diagnostics.append(row)
    if not accepted:
        raise ValueError('No completed Phase 4 outputs with verified dev logits/receipts; attach the saved training outputs')
    return accepted, diagnostics, dev_rows


def choose_recipe(candidates, rows, allow_pairs=False, margin=.002):
    """Fixed equal-weight pairs only; nested conditional gate with single fallback."""
    gold = np.asarray([r['gold'] for r in rows])
    values = [c['dev_probability'] for c in candidates]
    recipes = [(i,) for i in range(len(values))]
    if allow_pairs:
        recipes += [pair for pair in combinations(range(len(values)), 2)
                    if sum(candidates[i]['receipt']['parameter_report']['total_unique_parameters'] for i in pair) < 300_000_000]
    probabilities_by_recipe = [np.mean([values[i] for i in recipe], axis=0) for recipe in recipes]
    score = lambda probability, indices: metrics(gold[indices], probability[indices] >= .5)['macro_f1']
    full = np.arange(len(rows))
    scores = [score(p, full) for p in probabilities_by_recipe]
    best_single = max(range(len(values)), key=lambda i: (scores[i], -i))
    chosen = max(range(len(recipes)), key=lambda i: (scores[i], -len(recipes[i]), -i))
    conditional = conditional_single = None
    groups = [r.get('group', r['id']) for r in rows]
    folds = min(5, int(np.bincount(gold, minlength=2).min()), len(set(groups)))
    if allow_pairs and len(recipes[chosen]) > 1 and folds >= 2:
        oof, baseline = np.zeros(len(rows), dtype=int), np.zeros(len(rows), dtype=int)
        for train, heldout in StratifiedGroupKFold(folds, shuffle=True, random_state=781).split(full, gold, groups):
            selected = max(range(len(recipes)), key=lambda i: (score(probabilities_by_recipe[i], train), -len(recipes[i]), -i))
            single = max(range(len(values)), key=lambda i: (score(values[i], train), -i))
            oof[heldout] = probabilities_by_recipe[selected][heldout] >= .5
            baseline[heldout] = values[single][heldout] >= .5
        conditional = metrics(gold, oof)['macro_f1']
        conditional_single = metrics(gold, baseline)['macro_f1']
    if len(recipes[chosen]) > 1 and (conditional is None or scores[chosen] < scores[best_single] + margin or conditional < conditional_single + margin):
        chosen = best_single
    return recipes[chosen], probabilities_by_recipe[chosen], {
        'selected_indices': list(recipes[chosen]), 'raw_dev_macro_f1': scores[chosen],
        'best_single_raw_dev_macro_f1': scores[best_single],
        'conditional_recipe_OOF_macro_f1': conditional, 'conditional_single_OOF_macro_f1': conditional_single,
        'scope': 'Conditional dev selection, not independent test estimation; checkpoints already used dev',
        'pairs_enabled': allow_pairs, 'test_labels_used': False}


# CELL: Freeze recipe/threshold before reading any test logits
def select(input_root, prepared, output, explicit=None, allow_pairs=False):
    output = Path(output)
''')


[Markdown Cell]

Importable source part 3

In [54]:
# [Code Cell]
append_module('select.py', r'''    output.mkdir(parents=True, exist_ok=True)
    candidates, diagnostics, rows = completed_outputs(input_root, prepared, explicit)
    atomic_json(output / 'candidate_report.json', diagnostics)
    coverage = {(c['receipt']['family'], c['receipt']['method']) for c in candidates}
    required = {('a', 'full'), ('a', 'lorma'), ('b', 'bitfit'), ('c', 'adapters'), ('d', 'partial'), ('b', 'kd')}
    recipe, probability, report = choose_recipe(candidates, rows, allow_pairs)
    calibration = choose_threshold(probability, rows)
    selected = [candidates[i] for i in recipe]
    report.update(members=[c['receipt']['experiment_id'] for c in selected], calibration=calibration,
                  required_experiments_complete=required <= coverage, missing_methods=sorted(required - coverage))
    atomic_json(output / 'frozen_selection.json', report)
    dev = prediction(probability, rows, calibration['threshold'])
    atomic_json(output / 'dev_prediction.json', dev)
    official(Path(prepared), output / 'dev_prediction.json', calibration['selected_dev_macro_f1'], output)
    test_rows = list(records(Path(prepared) / 'test.jsonl'))
    expected = [{'id': r['id']} for r in test_rows]
    test_values = []
    for candidate in selected:
        root, receipt = candidate['root'], candidate['receipt']
        for name in ('test_logits.npy', 'test_order.json'):
            if sha256_file(root / 'predictions' / name) != receipt['logit_artifacts'][name]:
                raise ValueError('Changed test prediction artifact')
        if read_json(root / 'predictions/test_order.json') != expected:
            raise ValueError('Test original IDs/order differ')
        test_values.append(probabilities(np.load(root / 'predictions/test_logits.npy', allow_pickle=False)))
    path = package(prediction(np.mean(test_values, axis=0), test_rows, calibration['threshold']), test_rows,
                   output, 'submission_P4_Validation_Selected.zip')
    report.update(status='submission_verified', zip_path=str(path), zip_sha256=sha256_file(path), private_F1=None)
    atomic_json(output / 'final_report.json', report)
    return report


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument('--plan', required=True)
    args = parser.parse_args()
    print(select(**read_json(args.plan)), flush=True)


if __name__ == '__main__':
    main()
''')


[Markdown Cell]

### state.py

Two atomic recovery slots plus a separate best snapshot preserve model/Adam/scaler/scheduler and exact next case-update cursor. Checksummed receipts detect truncation; an intact older slot can recover. Preserve the full run output, not just a notebook or observations ZIP.

[Markdown Cell]

Importable source part 1

In [55]:
# [Code Cell]
append_module('state.py', r'''# CELL: Per-rank reproducibility and successful-update scheduling
import math
import os
from pathlib import Path
import random
import numpy as np
import torch
from .io_utils import atomic_json, ensure_space, publish_copy, read_json, sha256_file

def seed_everything(seed: int) -> None:
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False

def capture_rng(device: torch.device) -> dict:
    numpy_state = np.random.get_state()
    return {'python':random.getstate(), 'numpy':[numpy_state[0],numpy_state[1].tolist(),*numpy_state[2:]],
            'torch':torch.get_rng_state(), 'cuda':torch.cuda.get_rng_state(device) if device.type=='cuda' else None}

def restore_rng(state: dict, device: torch.device) -> None:
    random.setstate(state['python'])
    value=state['numpy']; np.random.set_state((value[0],np.asarray(value[1],dtype=np.uint32),*value[2:]))
    torch.set_rng_state(state['torch'])
    if device.type=='cuda':
        torch.cuda.set_rng_state(state['cuda'],device)

class WarmupCosine:
    """Schedule advances after successful optimizer updates, including on resume."""
    def __init__(self, optimizer, total_steps: int, warmup_fraction: float, minimum: float):
        self.optimizer, self.total_steps, self.minimum = optimizer,total_steps,minimum
        self.warmup_steps=max(1,int(total_steps*warmup_fraction))
        self.base_lrs=[group['lr'] for group in optimizer.param_groups]; self.steps=0
        self.apply()

    def apply(self) -> None:
        step=self.steps+1
        if step <= self.warmup_steps:
            scale=step/self.warmup_steps
        else:
            progress=min(1.0,(step-self.warmup_steps)/max(1,self.total_steps-self.warmup_steps))
            scale=self.minimum+(1-self.minimum)*0.5*(1+math.cos(math.pi*progress))
        for group,base in zip(self.optimizer.param_groups,self.base_lrs):
            group['lr']=base*scale

    def advance(self) -> None:
        self.steps+=1; self.apply()

''')


[Markdown Cell]

Importable source part 2

In [56]:
# [Code Cell]
append_module('state.py', r'''    def state_dict(self) -> dict:
        return dict(steps=self.steps,total_steps=self.total_steps,warmup_steps=self.warmup_steps,
                    minimum=self.minimum,base_lrs=self.base_lrs)

    def load_state_dict(self, value: dict) -> None:
        for key in ('total_steps','warmup_steps','minimum','base_lrs'):
            if value[key] != getattr(self,key):
                raise ValueError(f'Scheduler plan changed: {key}')
        self.steps=value['steps']; self.apply()

# CELL: Two atomic full-state recovery slots plus a validation-best snapshot
class Checkpoints:
    """Commit receipts last; select the latest intact state, preserving read-only inputs."""
    def __init__(self, run: Path, fingerprint: str, reserve: int, extra_roots: list[str] | None = None):
        self.directory=run/'checkpoints'; self.directory.mkdir(parents=True,exist_ok=True)
        self.fingerprint,self.reserve,self.generation=fingerprint,reserve,0
        self.roots=[self.directory,*[Path(root)/'checkpoints' for root in (extra_roots or [])]]
        self.warnings=[]

    def candidates(self, best_only: bool = False) -> list[tuple[Path,dict]]:
        result,seen=[],False
        for root in self.roots:
            for name in (('best.pt',) if best_only else ('slot_0.pt','slot_1.pt','best.pt')):
                path=root/name; sidecar=root/(name+'.json')
                if not path.exists() and not sidecar.exists():
                    continue
                seen=True
                try:
                    receipt=read_json(sidecar)
                    if receipt.get('fingerprint') != self.fingerprint or receipt.get('version') != 1:
                        raise ValueError('Experiment/source/Torch fingerprint mismatch')
                    if path.stat().st_size != receipt['bytes'] or sha256_file(path) != receipt['sha256']:
                        raise ValueError('Checkpoint size/SHA mismatch')
                    result.append((path,receipt)); self.generation=max(self.generation,receipt['generation'])
                except (ValueError,KeyError,OSError) as error:
                    self.warnings.append(f'{path}: {error}')
        if seen and not result:
            raise ValueError('All visible CJPE checkpoints failed verification. Refusing to silently restart. '+ '; '.join(self.warnings))
        return result

    def latest(self) -> tuple[Path,dict] | None:
        choices=self.candidates()
        if not choices:
            return None
        winner=max(choices,key=lambda pair:(pair[1]['step'],pair[1]['generation']))
        tied=[receipt['sha256'] for _,receipt in choices if (receipt['step'],receipt['generation'])==
              (winner[1]['step'],winner[1]['generation'])]
        if len(set(tied))>1:
''')


[Markdown Cell]

Importable source part 3

In [57]:
# [Code Cell]
append_module('state.py', r'''            raise ValueError('Conflicting recovery states at the same step. Attach one exact saved version.')
        return winner

    def best(self) -> tuple[Path,dict] | None:
        values=self.candidates(best_only=True)
        return max(values,key=lambda pair:(pair[1]['validation_macro_f1'],pair[1]['step'])) if values else None

    def adopt(self, candidate: tuple[Path,dict], best: bool = False) -> tuple[Path,dict]:
        source,receipt=candidate
        destination=self.directory/('best.pt' if best else f"slot_{receipt['generation']%2}.pt")
        if source.resolve()!=destination.resolve():
            ensure_space(self.directory,source.stat().st_size,self.reserve)
            publish_copy(source,destination)
            if sha256_file(destination)!=receipt['sha256']:
                raise ValueError('Restaged checkpoint checksum mismatch.')
            atomic_json(destination.with_name(destination.name+'.json'),receipt)
        return destination,receipt

    def load(self, candidate: tuple[Path,dict]) -> dict:
        path,receipt=candidate
        self.generation=max(self.generation,receipt['generation'])
        payload=torch.load(path,map_location='cpu',weights_only=True,mmap=True)
        required={'version','fingerprint','config','model','optimizer','scaler','scheduler','epoch','step','cursor','rng_by_rank'}
        if required-payload.keys() or payload['fingerprint']!=self.fingerprint:
            raise ValueError('Incomplete/incompatible CJPE checkpoint.')
        cursor=payload['cursor']
        if payload['epoch']!=receipt['epoch'] or payload['step']!=receipt['step'] or cursor['global_step']!=payload['step']:
            raise ValueError('Checkpoint cursor and receipt disagree.')
        if payload['scheduler']['steps']!=cursor['optimizer_steps'] or len(payload['rng_by_rank'])!=payload['config']['world_size']:
            raise ValueError('Missing/misaligned scheduler or per-rank RNG state.')
        return payload

    def commit(self, payload: dict, best: bool = False) -> tuple[Path,dict]:
        self.generation+=1
        path=self.directory/f'slot_{self.generation%2}.pt'
        temporary=path.with_name(path.name+'.partial')
        estimate=sum(tensor.numel()*tensor.element_size() for tensor in payload['model'].values())*3+(16<<20)
        ensure_space(self.directory,estimate,self.reserve)
        with temporary.open('wb') as stream:
            torch.save(payload,stream); stream.flush(); os.fsync(stream.fileno())
        os.replace(temporary,path)
        receipt={'version':1,'fingerprint':self.fingerprint,'sha256':sha256_file(path),'bytes':path.stat().st_size,
                 'epoch':payload['epoch'],'step':payload['step'],'generation':self.generation,
                 'validation_macro_f1':payload['cursor']['best_macro_f1']}
        atomic_json(path.with_name(path.name+'.json'),receipt)
        if best:
            publish_copy(path,self.directory/'best.pt')
            atomic_json(self.directory/'best.pt.json',receipt)
''')


[Markdown Cell]

Importable source part 4

In [58]:
# [Code Cell]
append_module('state.py', r'''        return path,receipt
''')


[Markdown Cell]

### train.py

$L=\sum_i w_{y_i}\,CE(z_i,y_i)/\sum_i w_{y_i}$. DDP scales each local loss by world size over the global real-case denominator. KD blends CE with $T^2 KL(p_t/T\Vert p_s/T)$ on training cases only. LR advances only after successful AMP updates. Both ranks save RNG at safe update boundaries.

[Markdown Cell]

Importable source part 1

In [59]:
# [Code Cell]
append_module('train.py', r'''# CELL: One global case-normalized supervised/KD loss, with successful-update scheduling
import argparse
from contextlib import nullcontext
from datetime import timedelta
import math
import os
from pathlib import Path
import signal
import socket
import time
import numpy as np
import torch
import torch.distributed as dist
import torch.multiprocessing as mp
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.nn import functional as F
from tqdm.auto import tqdm
from .config import CJPEConfig
from .data import CaseDataset, CasePlan, collate, training_weights
from .io_utils import atomic_json, read_json, run_lock
from .metrics import official
from .model import build_model, forward
from .performance import MicroBatches
from .predict import logits_partition, score_dev
from .state import Checkpoints, WarmupCosine, capture_rng, restore_rng, seed_everything


def objective(logits, targets, weights, teacher, cfg):
    scores = logits.float()
    supervised = F.cross_entropy(scores, targets, weight=weights, ignore_index=-100, reduction='sum')
    if cfg.method != 'kd':
        return supervised
    temperature = cfg.distillation_temperature
    probability = F.softmax(teacher.float() / temperature, dim=-1)
    divergence = F.kl_div(F.log_softmax(scores / temperature, dim=-1), probability, reduction='none').sum(-1) * temperature ** 2
    selected = weights[targets.clamp_min(0)] * (targets != -100)
    return (1 - cfg.distillation_weight) * supervised + cfg.distillation_weight * (divergence * selected).sum()


def optimizer_for(model, cfg):
    groups = [{'params': [p for n, p in model.named_parameters() if n.startswith('backbone.') and p.requires_grad], 'lr': cfg.backbone_lr},
              {'params': [p for n, p in model.named_parameters() if not n.startswith('backbone.') and p.requires_grad], 'lr': cfg.head_lr}]
    return torch.optim.AdamW(groups, weight_decay=cfg.weight_decay, fused=True)


def initial_cursor():
    return {'epoch': 0, 'next_batch': 0, 'global_step': 0, 'optimizer_steps': 0, 'history': [],
            'best_macro_f1': -1., 'bad_epochs': 0, 'complete': False, 'loss_sum': 0., 'weight_sum': 0.,
''')


[Markdown Cell]

Importable source part 2

In [60]:
# [Code Cell]
append_module('train.py', r'''            'cases': 0, 'chunks': 0, 'overflows': 0, 'consecutive_overflows': 0}


def save_all(rank, manager, model, optimizer, scaler, scheduler, cursor, cfg, job, device, best=False, paused=False):
    states = [None] * cfg.world_size if rank == 0 else None
    dist.gather_object(capture_rng(device), states, dst=0)
    if rank == 0:
        payload = {'version': 1, 'fingerprint': job['fingerprint'], 'config': job['config'],
                   'model': model.module.state_dict(), 'optimizer': optimizer.state_dict(), 'scaler': scaler.state_dict(),
                   'scheduler': scheduler.state_dict(), 'epoch': cursor['epoch'], 'step': cursor['global_step'],
                   'cursor': cursor, 'rng_by_rank': states}
        _, receipt = manager.commit(payload, best=best)
        summary = {'status': 'complete' if cursor['complete'] else ('paused_recoverable' if paused else 'in_progress'),
                   'epoch': cursor['epoch'], 'next_batch': cursor['next_batch'], 'step': cursor['global_step'],
                   'optimizer_steps': cursor['optimizer_steps'], 'best_dev_macro_f1': cursor['best_macro_f1'],
                   'history': cursor['history'], 'checkpoint_sha256': receipt['sha256'], 'fingerprint': job['fingerprint']}
        atomic_json(Path(cfg.run_dir) / 'training_summary.json', summary)
    dist.barrier()


def validation(rank, model, job, cfg, device):
    local = logits_partition(model, job, cfg, 'dev', device, rank, cfg.world_size)
    gathered = [None] * cfg.world_size if rank == 0 else None
    dist.gather_object(local, gathered, dst=0)
    value = None
    if rank == 0:
        joined = {key: values for shard in gathered for key, values in shard.items()}
        value, predictions = score_dev(joined, job)
        output = Path(cfg.run_dir)
        atomic_json(output / 'dev_prediction.json', predictions)
        official(Path(job['prepared_dir']), output / 'dev_prediction.json', value['macro_f1'], output)
        atomic_json(output / 'dev_metrics.json', value)
    wrapped = [value]
    dist.broadcast_object_list(wrapped, src=0)
    return wrapped[0]


# CELL: Two NCCL workers; exact next-case restart and collective timed-stop checkpoints
def worker(rank, job, port, event):
    cfg = CJPEConfig(**job['config'])
    cfg.validate()
    torch.set_num_threads(2)
    torch.cuda.set_device(rank)
    device = torch.device('cuda', rank)
    dist.init_process_group('nccl', init_method=f'tcp://127.0.0.1:{port}', rank=rank,
                            world_size=2, timeout=timedelta(hours=2))
    try:
        seed_everything(cfg.seed)
''')


[Markdown Cell]

Importable source part 3

In [61]:
# [Code Cell]
append_module('train.py', r'''        manager = Checkpoints(Path(cfg.run_dir), job['fingerprint'], cfg.min_free_bytes, job['resume_roots'])
        latest = (Path(job['resume_checkpoint']), read_json(job['resume_checkpoint'] + '.json')) if job['resume_checkpoint'] else None
        payload = manager.load(latest) if latest else None
        model = build_model(job, cfg, device, load_parent=payload is None)
        if payload:
            model.load_state_dict(payload['model'], strict=True)
        ddp = DDP(model, device_ids=[rank], broadcast_buffers=False, find_unused_parameters=False,
                  gradient_as_bucket_view=True)
        optimizer = optimizer_for(model, cfg)
        teacher = np.load(Path(cfg.run_dir) / 'teacher_cache.npy', mmap_mode='r', allow_pickle=False) if cfg.method == 'kd' else None
        dataset = CaseDataset(job['cache'], 'train', cfg, teacher)
        steps = math.ceil(len(dataset) / (cfg.world_size * cfg.micro_batch_size * cfg.accumulation_steps))
        scheduler = WarmupCosine(optimizer, steps * job['schedule_epochs'], cfg.warmup_fraction, cfg.min_lr_ratio)
        scaler = torch.amp.GradScaler('cuda', init_scale=4096)
        cursor = initial_cursor()
        if payload:
            scheduler.total_steps = payload['scheduler']['total_steps']
            scheduler.warmup_steps = payload['scheduler']['warmup_steps']
            scheduler.minimum = payload['scheduler']['minimum']
            scheduler.base_lrs = payload['scheduler']['base_lrs']
            optimizer.load_state_dict(payload['optimizer'])
            scaler.load_state_dict(payload['scaler'])
            scheduler.load_state_dict(payload['scheduler'])
            cursor = payload['cursor']
            restore_rng(payload['rng_by_rank'][rank], device)
            del payload
            if cursor['global_step'] != cursor['epoch'] * steps + cursor['next_batch'] or not 0 <= cursor['next_batch'] <= steps:
                raise ValueError('Recovery cursor differs from the exhaustive whole-case plan')
        else:
            seed_everything(cfg.seed + rank)
            save_all(rank, manager, ddp, optimizer, scaler, scheduler, cursor, cfg, job, device)
        if job['extension_requested'] and cursor['complete']:
            cursor['complete'] = False
            cursor['bad_epochs'] = 0
        if cursor['complete']:
            save_all(rank, manager, ddp, optimizer, scaler, scheduler, cursor, cfg, job, device)
            return
        weights_numpy = training_weights(job['cache'], cfg)
        weights = torch.tensor(weights_numpy, device=device)
        normalizers = np.asarray([weights_numpy[row['gold']] for row in dataset.rows] + [0.])
        started = time.monotonic()
        for epoch in range(cursor['epoch'], cfg.max_epochs):
            epoch_started = time.monotonic()
            ddp.train()
            plan = CasePlan(dataset.rows, cfg, rank, epoch)
            progress = tqdm(total=steps, initial=cursor['next_batch'],
                            desc=f'{cfg.experiment_id} epoch {epoch + 1}/{cfg.max_epochs}', disable=rank != 0)
            with MicroBatches(dataset, plan, cursor['next_batch'], cfg) as batches:
''')


[Markdown Cell]

Importable source part 4

In [62]:
# [Code Cell]
append_module('train.py', r'''                for step in range(cursor['next_batch'], steps):
                    stop = torch.tensor(int(event.is_set() or time.monotonic() - started >= cfg.max_run_seconds), device=device)
                    dist.all_reduce(stop, op=dist.ReduceOp.MAX)
                    if stop.item():
                        progress.close()
                        save_all(rank, manager, ddp, optimizer, scaler, scheduler, cursor, cfg, job, device, paused=True)
                        return
                    denominator = float(normalizers[plan.global_indices(step)].sum())
                    if denominator <= 0:
                        raise ValueError('A global update contains no real weighted cases')
                    optimizer.zero_grad(set_to_none=True)
                    stats = torch.zeros(3, device=device, dtype=torch.float64)
                    for micro in range(cfg.accumulation_steps):
                        batch = next(batches)
                        targets = batch['targets'].to(device, non_blocking=True)
                        context = ddp.no_sync() if micro + 1 < cfg.accumulation_steps else nullcontext()
                        with context:
                            with torch.autocast('cuda', dtype=torch.float16):
                                logits = forward(ddp, batch, device)
                                loss = objective(logits, targets, weights, batch['teacher'].to(device, non_blocking=True), cfg)
                            scaler.scale(loss * cfg.world_size / denominator).backward()
                        stats[0] += loss.detach().double()
                        stats[1] += (targets != -100).sum()
                        stats[2] += batch['input_ids'].shape[0]
                    dist.all_reduce(stats)
                    loss_sum, case_count, chunk_count = stats.cpu().tolist()
                    if not all(math.isfinite(value) for value in (loss_sum, case_count, chunk_count)):
                        raise FloatingPointError('Nonfinite loss; resume the last committed checkpoint')
                    scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(ddp.parameters(), cfg.max_grad_norm)
                    previous_scale = scaler.get_scale()
                    scaler.step(optimizer)
                    scaler.update()
                    if scaler.get_scale() >= previous_scale:
                        scheduler.advance()
                        cursor['optimizer_steps'] += 1
                        cursor['consecutive_overflows'] = 0
                    else:
                        cursor['overflows'] += 1
                        cursor['consecutive_overflows'] += 1
                    if cursor['consecutive_overflows'] > cfg.max_consecutive_overflows:
                        raise FloatingPointError('Repeated AMP overflows; inspect committed state before continuing')
                    cursor['global_step'] += 1
                    cursor['next_batch'] = step + 1
                    cursor['loss_sum'] += loss_sum
                    cursor['weight_sum'] += denominator
                    cursor['cases'] += int(case_count)
                    cursor['chunks'] += int(chunk_count)
''')


[Markdown Cell]

Importable source part 5

In [63]:
# [Code Cell]
append_module('train.py', r'''                    optimizer.zero_grad(set_to_none=True)
                    if rank == 0:
                        progress.update()
                        if cursor['global_step'] % cfg.log_every_steps == 0 or step + 1 == steps:
                            progress.set_postfix(loss=cursor['loss_sum'] / cursor['weight_sum'], lr=optimizer.param_groups[0]['lr'],
                                                 step=cursor['global_step'], cases=cursor['cases'], amp_skips=cursor['overflows'])
                    if cursor['global_step'] % 500 == 0:
                        save_all(rank, manager, ddp, optimizer, scaler, scheduler, cursor, cfg, job, device)
            progress.close()
            save_all(rank, manager, ddp, optimizer, scaler, scheduler, cursor, cfg, job, device)
            value = validation(rank, ddp.module, job, cfg, device)
            if cursor['cases'] != len(dataset):
                raise ValueError('Epoch failed exact once-per-original-case supervision')
            improved = value['macro_f1'] > cursor['best_macro_f1'] + 1e-12
            cursor['best_macro_f1'] = max(cursor['best_macro_f1'], value['macro_f1'])
            cursor['bad_epochs'] = 0 if improved else cursor['bad_epochs'] + 1
            cursor['history'].append({'epoch': epoch + 1, 'weighted_objective': cursor['loss_sum'] / cursor['weight_sum'],
                                      'training_cases': cursor['cases'], 'training_chunks_with_masked_dummies': cursor['chunks'],
                                      'overflows': cursor['overflows'], 'epoch_seconds_in_this_session_including_validation': time.monotonic() - epoch_started, **value})
            cursor.update(epoch=epoch + 1, next_batch=0, loss_sum=0., weight_sum=0., cases=0, chunks=0, overflows=0)
            cursor['complete'] = cursor['epoch'] >= cfg.max_epochs or (cursor['epoch'] >= cfg.minimum_epochs and cursor['bad_epochs'] >= cfg.patience)
            save_all(rank, manager, ddp, optimizer, scaler, scheduler, cursor, cfg, job, device, best=improved)
            if rank == 0:
                print({'epoch': cursor['epoch'], 'native_dev_macro_F1': value['macro_f1'], 'complete': cursor['complete']}, flush=True)
            if cursor['complete']:
                return
        dataset.close()
    finally:
        dist.destroy_process_group()


# CELL: Importable spawn entrypoint, never nn.DataParallel or notebook-local worker functions
def main():
    parser = argparse.ArgumentParser()
    parser.add_argument('--job', required=True)
    args = parser.parse_args()
    job = read_json(args.job)
    cfg = CJPEConfig(**job['config'])
    from .prepare import require_hardware
    require_hardware()
    cfg.validate()
    os.environ.setdefault('TOKENIZERS_PARALLELISM', 'false')
    os.environ.setdefault('TORCH_NCCL_ASYNC_ERROR_HANDLING', '1')
    with run_lock(Path(cfg.run_dir)):
        manager = Checkpoints(Path(cfg.run_dir), job['fingerprint'], cfg.min_free_bytes, job['resume_roots'])
        latest, best = manager.latest(), manager.best()
        if best:
            manager.adopt(best, best=True)
''')


[Markdown Cell]

Importable source part 6

In [64]:
# [Code Cell]
append_module('train.py', r'''        if latest:
            latest = manager.adopt(latest)
            job['resume_checkpoint'] = str(latest[0])
            print('Resume exact saved step', latest[1]['step'], 'epoch', latest[1]['epoch'], flush=True)
        else:
            job['resume_checkpoint'] = None
        context = mp.get_context('spawn')
        event = context.Event()
        signal.signal(signal.SIGINT, lambda *_: event.set())
        signal.signal(signal.SIGTERM, lambda *_: event.set())
        with socket.socket() as listener:
            listener.bind(('127.0.0.1', 0))
            port = listener.getsockname()[1]
        processes = mp.spawn(worker, args=(job, port, event), nprocs=2, join=False)
        while not processes.join(timeout=1):
            pass


if __name__ == '__main__':
    main()
''')


[Markdown Cell]

### upgrade.py

An explicit checked engine upgrade copies model/Adam/scaler/scheduler/cursor and both RNG states from a known original full Phase 4 output into a new experiment ID. The original is untouched. Kernel/batching changes need not reproduce the old bitwise trajectory.

[Markdown Cell]

Importable source part 1

In [65]:
# [Code Cell]
append_module('upgrade.py', r'''# CELL: Explicit audited engine upgrade, preserving tensors, Adam and the data cursor
from dataclasses import asdict
from pathlib import Path
import torch
from .config import CJPEConfig
from .io_utils import atomic_json, identity, read_json
from .legacy_contract import LEGACY_SOURCE_SHA256, FAST_V1_SOURCE_SHA256
from .parents import source_hashes
from .state import Checkpoints

PERFORMANCE_CHOICES = {'experiment_id', 'chunk_micro_batch', 'checkpoint_encoder', 'prefetch_batches',
                       'log_every_steps', 'inference_cases_per_batch', 'inference_chunk_budget', 'frozen_prefix_cache_tokens'}
LEGACY_EXCLUSIONS = {'run_dir', 'input_root', 'prepared_dir', 'parent_run_dir', 'tokenizer_path', 'metadata_path',
                     'resume_run_dir', 'teacher_run_dir', 'expected_parent_sha256', 'parent_checkpoint',
                     'max_epochs', 'target_total_epochs', 'max_run_seconds', 'min_free_bytes'}


def upgrade_candidate(cfg):
    """Accept only our known complete source contracts, never an arbitrary pickle."""
    root = Path(cfg.upgrade_run_dir)
    saved = read_json(root / 'job.json')
    prior = CJPEConfig(**saved['config'])
    actual = source_hashes(root / 'code/cjpe')
    current = source_hashes(Path(__file__).parent)
    if actual != saved['runtime_source_sha256'] or actual not in (LEGACY_SOURCE_SHA256, FAST_V1_SOURCE_SHA256, current):
        raise ValueError('Upgrade needs the complete known original/optimized Phase 4 source, not observations or a changed runtime')
    if saved.get('version') != 'cs781-cjpe-v1' or saved['environment']['torch'] != str(torch.__version__):
        raise ValueError('Upgrade requires the same CJPE protocol and exact saved PyTorch build')
    comparable = lambda c: {k:v for k,v in c.choices().items() if k not in PERFORMANCE_CHOICES}
    if comparable(prior) != comparable(cfg) or prior.experiment_id == cfg.experiment_id:
        raise ValueError('Use a NEW fast experiment ID and preserve all learning choices, global case batch, chunk length and head architecture')
    if source_hashes(root / 'parent_source' / ('model_' + cfg.family)) != saved['parent_contract']['source_sha256']:
        raise ValueError('Upgrade foundation source snapshot changed')
    choices = ({k:v for k,v in saved['config'].items() if k not in LEGACY_EXCLUSIONS}
               if actual == LEGACY_SOURCE_SHA256 else prior.choices())
    expected = identity({'version': saved['version'], 'choices': choices, 'prepared': saved['prepared_identity'],
        'parent': saved['parent']['checkpoint_sha256'], 'tokenizer': saved['parent']['tokenizer_sha256'],
        'runtime': saved['runtime_source_sha256'], 'environment': saved['environment'],
        'cache': saved['cache']['identity'], 'case_metadata': saved['cache']['manifest_sha256'], 'teacher': saved.get('teacher')})
    if expected != saved['fingerprint']:
        raise ValueError('Cannot reproduce the original Phase 4 learning/data/source fingerprint')
    return root, {**saved, '_engine_upgrade': True}


def import_upgrade(job, previous, cfg):
    """Re-envelope unchanged tensor/state payloads under the new engine identity."""
    old_root, saved = previous
    if saved['prepared_identity'] != job['prepared_identity'] or saved['cache']['manifest_sha256'] != job['cache']['manifest_sha256']:
''')


[Markdown Cell]

Importable source part 2

In [66]:
# [Code Cell]
append_module('upgrade.py', r'''        raise ValueError('Upgrade cannot change prepared data, token cache or example order')
    original = Checkpoints(old_root, saved['fingerprint'], 0)
    latest, best = original.latest(), original.best()
    if not latest:
        raise FileNotFoundError('Upgrade requires actual full checkpoint files and matching receipts')
    manager = Checkpoints(Path(cfg.run_dir), job['fingerprint'], cfg.min_free_bytes)
    for candidate, is_best in ((best, True), (latest, False)):
        if not candidate:
            continue
        payload = original.load(candidate)
        if payload['cursor']['global_step'] != payload['step']:
            raise ValueError('Upgrade cursor/step mismatch')
        effective_config = asdict(cfg)
        effective_config['max_epochs'] = payload['config']['max_epochs']
        payload.update(fingerprint=job['fingerprint'], config=effective_config)
        manager.commit(payload, best=is_best)
        del payload
    imported_best = manager.best()
    report = {'status': 'upgraded_full_state', 'source_experiment': saved['config']['experiment_id'],
        'source_fingerprint': saved['fingerprint'], 'new_fingerprint': job['fingerprint'],
        'source_latest_sha256': latest[1]['sha256'], 'source_step': latest[1]['step'],
        'imported_best_sha256': imported_best[1]['sha256'] if imported_best else None,
        'model_optimizer_scaler_scheduler_cursor_and_per_rank_RNG_preserved': True,
        'global_case_batch_and_all_remaining_examples_preserved': True,
        'scope': 'Explicit engine transition, not bitwise continuation across different kernels/chunk microbatches. Original input remains unchanged.'}
    atomic_json(Path(cfg.run_dir) / 'engine_upgrade_report.json', report)
    return report
''')


[Markdown Cell]

### workflow.py

Automatic input checks, actual full-size shape/gradient sanity, training, completion-aware export and small observations. Paused or failed runs cannot claim a final submission. Keep full saved Kaggle output for continuation or KD.

[Markdown Cell]

Importable source part 1

In [67]:
# [Code Cell]
append_module('workflow.py', r'''# CELL: Explicit completion, timed recovery and always-available observations
import argparse
from dataclasses import asdict
from pathlib import Path
import subprocess
import sys
import traceback
import zipfile
from .config import CJPEConfig
from .io_utils import atomic_json, read_json
from .prepare import prepare_job, require_hardware, sanity
from .state import Checkpoints


def stage_completed_export(job):
    """Publish a read-only completed input into the new writable replay run."""
    cfg = CJPEConfig(**job['config'])
    root = Path(cfg.run_dir)
    manager = Checkpoints(root, job['fingerprint'], cfg.min_free_bytes, job['resume_roots'])
    latest, best = manager.latest(), manager.best()
    if not latest or not best:
        raise FileNotFoundError('Export requires an actual complete latest checkpoint and best checkpoint')
    payload = manager.load(latest)
    cursor = payload['cursor']
    if not cursor['complete']:
        raise ValueError('Export cannot bypass an incomplete/paused training cursor')
    manager.adopt(best, best=True)
    latest = manager.adopt(latest)
    atomic_json(root / 'training_summary.json', {
        'status': 'complete', 'epoch': cursor['epoch'], 'next_batch': cursor['next_batch'],
        'step': cursor['global_step'], 'optimizer_steps': cursor['optimizer_steps'],
        'best_dev_macro_f1': cursor['best_macro_f1'], 'history': cursor['history'],
        'checkpoint_sha256': latest[1]['sha256'], 'fingerprint': job['fingerprint']})


def observations(root: Path):
    """Small diagnosis archive; deliberately not a replacement for model weights."""
    root.mkdir(parents=True, exist_ok=True)
    path = root / 'observations.zip'
    with zipfile.ZipFile(path.with_suffix('.zip.partial'), 'w', zipfile.ZIP_DEFLATED) as zipped:
        for item in sorted(root.rglob('*')):
            if item.is_file() and item.suffix in ('.json', '.log', '.txt') and item.stat().st_size <= 5 << 20:
                zipped.write(item, str(item.relative_to(root)))
    path.with_suffix('.zip.partial').replace(path)


def execute(plan_path, catalog_path):
    plan = read_json(plan_path)
''')


[Markdown Cell]

Importable source part 2

In [68]:
# [Code Cell]
append_module('workflow.py', r'''    cfg = CJPEConfig(**plan['config'])
    root = Path(cfg.run_dir)
    root.mkdir(parents=True, exist_ok=True)
    try:
        # Find/validate all input identities before the expensive full-size GPU checks.
        job = prepare_job(cfg, read_json(catalog_path))
        atomic_json(root / 'input_report.json', {
            'prepared': job['prepared_dir'], 'parent': job['parent']['checkpoint_sha256'],
            'training_cases': len(job['cache']['cases']['train']),
            'all_training_chunks': sum(r['chunks'] for r in job['cache']['cases']['train']),
            'resume_checkpoint': job['resume_checkpoint'], 'teacher': job.get('teacher'),
            'learning_choices': CJPEConfig(**job['config']).choices()})
        require_hardware()
        if plan.get('mode', 'train') == 'profile':
            from .profile import run_profile
            return run_profile(root / 'job.json')
        if plan.get('mode', 'train') not in ('train', 'export'):
            raise ValueError('Mode must be train, profile or export')
        if plan.get('mode', 'train') == 'train':
            sanity(job, CJPEConfig(**job['config']))
            subprocess.check_call([sys.executable, '-u', '-m', 'cjpe.train', '--job', str(root / 'job.json')])
        else:
            stage_completed_export(job)
        if read_json(root / 'training_summary.json')['status'] != 'complete':
            result = {'status': 'paused_recoverable', 'next_action': 'Save the complete output and attach it to this same notebook; resume the exact cursor. No final ZIP is claimed.'}
        else:
            from .predict import export
            result = export(root / 'job.json')
        atomic_json(root / 'final_report.json', result)
        return result
    except Exception as error:
        atomic_json(root / 'failure_report.json', {'status': 'failed', 'type': type(error).__name__,
            'message': str(error), 'traceback': traceback.format_exc(), 'action': 'Keep full checkpoints and consult input/teacher/failure reports; never delete committed recovery files.'})
        raise
    finally:
        observations(root)


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument('--plan', required=True)
    parser.add_argument('--catalog', required=True)
    args = parser.parse_args()
    print(execute(args.plan, args.catalog), flush=True)


if __name__ == '__main__':
    main()
''')


In [69]:
# [Code Cell]
EXPECTED_SOURCE_SHA256={'__init__.py': '746816a2d6168979d1ab82a8ee3df4e15f9e1f02b5816e683799fd5723dec47c', 'adaptation.py': '62ecd0842a2c12254d4b1bb86ab100bd7392846c4f60c38b4cdc48b6321c1ad7', 'analysis.py': '36eaafc33d1f51dad9941c304f39fa6e525c5dd4bd804f75f8f002644f734be3', 'config.py': '08b73f7ed2e487398e51a66799a495e70051537473a5fdc3283da88371c8d073', 'data.py': '644725ec8027965a23754810ecd8c3e33018b1837b17af6f5deecbbca3c55a34', 'io_utils.py': '3f9d828342c819690d390f0280c01f9eda160aab59c9cf424d6b3c2a31813e73', 'legacy_contract.py': 'f5fad17cdd00673ee87d01195973a195037b14471da1f40b4ca76b84603e9013', 'metrics.py': '8b1c074208775a9a64998567df8591e78045f3f5c38b99dce8bd0663e411a9db', 'model.py': '8692cc27cf9531abaf198f3d54dab57fb2eb9c5100939141ef509d205b6360c9', 'parents.py': 'a41e6eacf9f053177c4e934ac547138a60d9cd1084e5f53948354a79b97047e9', 'performance.py': 'bd3bfbaa476107db8c1896dbe5842055af2038dc0de36c8e427f0ee710f40ebf', 'predict.py': 'ab06d80b649a8f6fffbb8d6b8486ace44363bfc9cf02d0a0deada09db1697175', 'preparation.py': '33a5e84c6576e6815eb53a767adcb8104f1aeb37f52a7d42e0d2d15f259bb0bf', 'prepare.py': 'e2d10c17771adefb3c117963303483e2b5c6423bd4dae795d8238eda289012d5', 'profile.py': '7fe917c7a7d6fd9c453d76b9b4eeeb74610f7ccf7f0de8ab13793169229d1e9a', 'select.py': 'd0c4c1bc6731f5b311bcc9ac22dd9db6712da8fa54b5ad52f48bc0957aae306d', 'state.py': '369c7fd7d7e4ff16331f2f2f0a6348fd129c38c20bdbbd77fae786688bca0573', 'train.py': 'c31bce7a2e46658cde79f30c4a842f6f01486ea800c8ef86688a40701cb458c0', 'upgrade.py': 'ad5011a6d6d46eb6f4d55807fb8f554c5aa974e0b541bf604e649708335466d1', 'workflow.py': '86f007570a1b301749b4ea4a7c6dd88677b4dca8643b26210839ca650b5f00cd'}


[Markdown Cell]

## 3. Materialize exact importable runtime

In [70]:
# [Code Cell]
import hashlib
import importlib
code_directory=Path(OUTPUT_ROOT)/'code'
package_directory=code_directory/'cjpe'
package_directory.mkdir(parents=True,exist_ok=True)
for name,parts in module_parts.items():
    source=''.join(parts)
    assert hashlib.sha256(source.encode()).hexdigest()==EXPECTED_SOURCE_SHA256[name],name
    compile(source,name,'exec')
    destination=package_directory/name
    if (Path(OUTPUT_ROOT)/'job.json').exists() and destination.exists() and destination.read_bytes()!=source.encode():
        raise ValueError('Saved task runtime changed: use its exact notebook, or use a new experiment ID.')
    destination.write_bytes(source.encode())
for name in list(sys.modules):
    if name=='cjpe' or name.startswith('cjpe.'):
        del sys.modules[name]
sys.path.insert(0,str(code_directory))
importlib.invalidate_caches()


In [71]:
# [Code Cell]
PARENT_CATALOG={'a': {'family': 'a', 'runtime_version': 'cs781-model-a-v1', 'source_sha256': {'config.py': 'c0d1e9e7eb49a11e12ff0d7014cd07c6b90149460da5db3b06e9af71fbb3f390', 'model.py': '05118a8c9eec3e58adb272554e7356e81fb28c95cbe7905f39ebd5ee2d670319', 'data.py': '39b20ec14101e66b6c1f81464738161a71755f53f30fec7edb93deb7f4ffe9a8', 'state.py': '723193f75a39d3b8f16af1e6f1bfbd507b1775a0cd2430583d211e54e69d63c2', 'train.py': '3ea14c27742ab82e686083a669823678529d0f8beb46b2e6b2113f0aef452743', 'infer.py': '1cb8e96f2d49aa1c0b25c8e0ef6420c6d63f17ec8b72813d69b646748fb46013', 'sanity.py': '7716a61318767f9001ac08cdcafbe228f53c3c0d6f8fb9cc3806353ec2d6577f', '__init__.py': '59a82a09276925aa3342bc0991d79d474388cc6119bae880ced9d2403eb20a45'}, 'fingerprint_exclusions': ['data_dir', 'run_dir', 'resume_dir', 'initial_checkpoint', 'max_run_seconds', 'min_free_bytes', 'max_checkpoint_bytes', 'generation_examples_per_source', 'generation_token_limit'], 'data_fingerprint': '3571562850214ac1b423548cb0020fdcf1a3ffd8956e0368c456ec1b100292f4', 'audited_original_epochs': 2, 'audited_original_step': 29942, 'audited_original_fingerprint': '24f90bc6e148ffb53dcf6132138b69a95ced2e6a5ca34690601d29def059cddd', 'recorded_torch_candidates': ['2.11.0+cu128', '2.10.0+cu128'], 'class_name': 'LegalCausalDecoder'}, 'b': {'family': 'b', 'runtime_version': 'cs781-model-b-v1', 'source_sha256': {'config.py': '71407468d5c4b105b5db2c533fb736eb847b4b67cdf9d4f2ab014436adef1e9b', 'model.py': '09baae8e7ebe0508fb149a64c187975b7a86ae62376b5bc2ec4cef2fb2471a99', 'data.py': 'd01b01d43f5c4c24bb0429f7bbb8bbbd4c52d52c8787bd5ef3f40bdac28df919', 'state.py': '026036ddbdf43e949755da263eb97378855a28281129c6a63f3b3ea69abbe088', 'train.py': 'f928bb52eaafd17dc061d3003326fd13cf2f79b5ccbc226f6af0177429aefab0', 'infer.py': '80ee8fff95675dccb59cf19da091355020495c0f41ea16fdb17d8f24357a59f4', 'sanity.py': '6b1b96bc16a4d7f0945e139fd60f35813b0bf9d2b075c4fd893a87d432fc9212', '__init__.py': '3dfc8da18d4c5d73d0ff1b52dc572672f8095b605dcc86836832d77607749087'}, 'fingerprint_exclusions': ['data_dir', 'run_dir', 'resume_dir', 'max_run_seconds', 'min_free_bytes', 'max_checkpoint_bytes', 'generation_examples_per_source', 'generation_token_limit'], 'data_fingerprint': '3571562850214ac1b423548cb0020fdcf1a3ffd8956e0368c456ec1b100292f4', 'audited_original_epochs': 3, 'audited_original_step': 24267, 'audited_original_fingerprint': 'e686c0f98fb5fe8fc03553d0151961dd6af0dd52ac041753de0502521a922bb1', 'recorded_torch_candidates': ['2.11.0+cu128', '2.10.0+cu128'], 'class_name': 'LegalEncoderDecoder'}, 'c': {'family': 'c', 'runtime_version': 'cs781-model-c-v1', 'source_sha256': {'config.py': 'f9fea0bc87f74ccac761f7d4dc574c4d1315a459523dba21849279c17bc24ace', 'model.py': '6f91835c789d12aadb5355bc06050b8b6c1fc231cd16126fc39f31769706aad4', 'data.py': '39b20ec14101e66b6c1f81464738161a71755f53f30fec7edb93deb7f4ffe9a8', 'state.py': 'b3a3ed7109fdfe43ebc8e75db89f2916d33e95ebe901e99d3e4aa9b6e8422ee4', 'train.py': '9bf6cb023ad1c8fdc4fbb652b4b3b6d3551ae777df139184051f8d23f4e5c0c9', 'infer.py': '062010c87b66c7badcef8f7a62ef3a89fdc7f96149d317214b13a7e679f40b6a', 'sanity.py': 'c56b6ad692b90cb5afb23a03a75a26dde9d9b2c9ef3e14ca2ed37cb8b0f18a68', '__init__.py': '59a82a09276925aa3342bc0991d79d474388cc6119bae880ced9d2403eb20a45'}, 'fingerprint_exclusions': ['data_dir', 'run_dir', 'resume_dir', 'initial_checkpoint', 'max_run_seconds', 'min_free_bytes', 'max_checkpoint_bytes', 'generation_examples_per_source', 'generation_token_limit'], 'data_fingerprint': '092deede43ffd17c00e5d11e1f4aa3a443b7939af4504c488426b70c53d1ed54', 'audited_original_epochs': 1, 'audited_original_step': 7383, 'audited_original_fingerprint': '8febd61da7f1d32f78eee1cd4a865d72e5a5b3a0068739210db323859c6d7caa', 'recorded_torch_candidates': ['2.11.0+cu128', '2.10.0+cu128'], 'class_name': 'LegalTransformerGRU'}, 'd': {'family': 'd', 'runtime_version': 'cs781-model-d-v1', 'source_sha256': {'config.py': '42bd096bdf40822a7cb1d84e7f437e3e64d2ef20b99ee81c884f60639378e63a', 'model.py': 'a2300d34524df50cb43a3ff700e72b3a8764f7566d66ba5065c048c5f115eae9', 'data.py': '39b20ec14101e66b6c1f81464738161a71755f53f30fec7edb93deb7f4ffe9a8', 'state.py': 'dee5a29320756f4b84ec349b24091180be65a8f4c86e7f16eb74343737f4e2db', 'train.py': 'efb53d8eb54a08ef1a4abc49d5437eb472ebf5047bf790cf2f041c17fd6cef37', 'infer.py': 'aa3b849b0095bbdb5d81d3bfafe6a8134d4481856ee7cb9db5fba2bd84b80b67', 'sanity.py': '8c02e1c1a1e521a5d1e83fd96b3cfba3fa9f7c76c5824f62d45297a4829ca298', '__init__.py': 'f49e11a2d96bae160f02704fd5818fe613e77534e06a0f16150dbc4bc2d4f5e6'}, 'fingerprint_exclusions': ['data_dir', 'run_dir', 'resume_dir', 'initial_checkpoint', 'max_run_seconds', 'min_free_bytes', 'max_checkpoint_bytes', 'generation_examples_per_source', 'generation_token_limit'], 'data_fingerprint': '3a4dc2be3ff16945069b15d7a2e351012d4a0c9319b95b5e4534cc6596e8693f', 'audited_original_epochs': 1, 'audited_original_step': 1828, 'audited_original_fingerprint': 'b9659899c46424ec56ad0607f32bd6e7097752a8066dd30a38764469e05c1352', 'recorded_torch_candidates': ['2.11.0+cu128', '2.10.0+cu128'], 'class_name': 'LegalSparseDecoder'}}


[Markdown Cell]

## 4. Build the launch plan

In [72]:
# [Code Cell]
from cjpe.config import CJPEConfig
from cjpe.io_utils import atomic_json,read_json
SETTINGS=dict(experiment_id=EXPERIMENT_ID,family=FAMILY,method=METHOD,run_dir=OUTPUT_ROOT,
    prepared_dir=PREPARED_DIR,parent_run_dir=PARENT_RUN_DIR,tokenizer_path=TOKENIZER_PATH,
    metadata_path=METADATA_PATH,resume_run_dir=RESUME_RUN_DIR,teacher_run_dir=TEACHER_RUN_DIR,
    expected_parent_sha256=EXPECTED_PARENT_SHA256,max_epochs=MAX_EPOCHS,
    target_total_epochs=TARGET_TOTAL_EPOCHS,max_run_seconds=int(TRAINING_MINUTES*60),
    chunk_tokens=CHUNK_TOKENS,chunk_micro_batch=CHUNK_MICRO_BATCH,
    backbone_lr=BACKBONE_LR,head_lr=HEAD_LR,upgrade_run_dir=UPGRADE_FROM_RUN_DIR,
    prefetch_batches=PREFETCH_BATCHES,log_every_steps=LOG_EVERY_STEPS,
    inference_cases_per_batch=INFERENCE_CASES_PER_BATCH,inference_chunk_budget=INFERENCE_CHUNK_BUDGET)
if UPGRADE_FROM_RUN_DIR:
    previous_job=read_json(Path(UPGRADE_FROM_RUN_DIR)/'job.json')
    requested=SETTINGS.copy()
    SETTINGS=previous_job['config'].copy()
    override_names={'experiment_id','run_dir','prepared_dir','parent_run_dir','tokenizer_path',
        'metadata_path','resume_run_dir','teacher_run_dir','upgrade_run_dir','target_total_epochs',
        'max_run_seconds','chunk_micro_batch','prefetch_batches','log_every_steps',
        'inference_cases_per_batch','inference_chunk_budget'}
    SETTINGS.update({name:value for name,value in requested.items() if name in override_names})
    SETTINGS['input_root']='/kaggle/input'
    SETTINGS['expected_parent_sha256']=previous_job['parent']['checkpoint_sha256']
    print('Copied original learning/head/chunk/global-batch choices for the checked engine transition.')
CJPEConfig(**SETTINGS).validate()
plan_path=Path(OUTPUT_ROOT)/'pipeline_plan.json'
catalog_path=code_directory/'parent_catalog.json'
atomic_json(plan_path,{'mode':MODE,'config':SETTINGS})
atomic_json(catalog_path,PARENT_CATALOG)
print('Input root is /kaggle/input. Explicit settings:',SETTINGS)


Input root is /kaggle/input. Explicit settings: {'experiment_id': 'P4_01_A_Full_FAST_V2', 'family': 'a', 'method': 'full', 'run_dir': '/kaggle/working/cs781_phase4/P4_01_A_Full_FAST_V2', 'prepared_dir': None, 'parent_run_dir': None, 'tokenizer_path': None, 'metadata_path': None, 'resume_run_dir': None, 'teacher_run_dir': None, 'expected_parent_sha256': '0746fa1981f804a246b65d82f1d13e35c3a6bdf554bfff69df26a35dd1d5d131', 'max_epochs': 4, 'target_total_epochs': None, 'max_run_seconds': 18000, 'chunk_tokens': 256, 'chunk_micro_batch': 8, 'backbone_lr': 2e-05, 'head_lr': 0.0002, 'upgrade_run_dir': None, 'prefetch_batches': True, 'log_every_steps': 10, 'inference_cases_per_batch': 4, 'inference_chunk_budget': 64}


[Markdown Cell]

## 5. Stream progress and stop safely

In [73]:
# [Code Cell]
import codecs
import select
import signal

def run_streamed(command):
    environment=os.environ.copy()
    environment['PYTHONPATH']=str(code_directory)+os.pathsep+environment.get('PYTHONPATH','')
    process=subprocess.Popen(command,cwd=code_directory,env=environment,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,bufsize=0)
    decoder=codecs.getincrementaldecoder('utf-8')(errors='replace')
    assert process.stdout is not None
    try:
        while True:
            try:
                ready,_,_=select.select([process.stdout],[],[],1.0)
                if ready:
                    chunk=os.read(process.stdout.fileno(),8192)
                    if chunk:
                        sys.stdout.write(decoder.decode(chunk));sys.stdout.flush()
                    elif process.poll() is not None:
                        break
                elif process.poll() is not None:
                    break
            except KeyboardInterrupt:
                print('Requesting a saved stop after the current update. Keep this cell running.')
                process.send_signal(signal.SIGINT)
        status=process.wait()
        sys.stdout.write(decoder.decode(b'',final=True));sys.stdout.flush()
        if status:
            raise subprocess.CalledProcessError(status,command)
    finally:
        process.stdout.close()


[Markdown Cell]

## 6. Verify, profile or train and export

In [74]:
# [Code Cell]
run_streamed([sys.executable,'-u','-m','cjpe.workflow','--plan',str(plan_path),'--catalog',str(catalog_path)])


Tokenize test: 100%|██████████| 1517/1517 [00:19<00:00, 77.70it/s] 
GPU 0 : Tesla T4
GPU 1 : Tesla T4
GPU 0 : Tesla T4
GPU 1 : Tesla T4
Resume exact saved step 17260 epoch 3
P4_01_A_Full_FAST_V2 epoch 4/4:  74%|███████▍  | 3478/4674 [06:30<28:36,  1.44s/it, amp_skips=2, cases=27824, loss=0.181, lr=2.23e-6, step=17500]/usr/local/lib/python3.13/dist-packages/torch/distributed/c10d_logger.py:83: UserWarning: barrier(): using the device under current context. You can specify `device_id` in `init_process_group` to mute this warning.
  return func(*args, **kwargs)
Predict dev: 100%|██████████| 994/994 [01:51<00:00,  8.92cases/s]
{'epoch': 4, 'native_dev_macro_F1': 0.7273431755484749, 'complete': True}
Predict test: 100%|██████████| 759/759 [01:27<00:00,  8.63cases/s]
{'status': 'submission_verified', 'version': 'cjpe-output-v1', 'experiment_id': 'P4_01_A_Full_FAST_V2', 'family': 'a', 'method': 'full', 'class_order': [0, 1], 'prepared_identity': '1e3bb9cbdd9efafc308804590efd608567da562a3e6ce0

[Markdown Cell]

## 7. Read the real status and keep the correct files

In [75]:
# [Code Cell]
from IPython.display import FileLink,display
report_name='profile_report.json' if MODE=='profile' else 'final_report.json'
report=read_json(Path(OUTPUT_ROOT)/report_name)
print(json.dumps(report,indent=2))
if report.get('status')=='submission_verified':
    display(FileLink(report['zip_path']))
if (Path(OUTPUT_ROOT)/'observations.zip').exists():
    display(FileLink(str(Path(OUTPUT_ROOT)/'observations.zip')))
print('Preserve the COMPLETE saved output for recovery/KD. An observations ZIP has no model weights.')


{
  "status": "submission_verified",
  "version": "cjpe-output-v1",
  "experiment_id": "P4_01_A_Full_FAST_V2",
  "family": "a",
  "method": "full",
  "class_order": [
    0,
    1
  ],
  "prepared_identity": "1e3bb9cbdd9efafc308804590efd608567da562a3e6ce0774f34ca254bcb24ab",
  "fingerprint": "b01fbedcbab8662a5ebe836925ea7b6656ff78b245189825f7056ce9e775f7c6",
  "checkpoint_sha256": "863087f559196ea95ea69b488d7a440669a3bc72e009aa84a79696532c4b1d39",
  "checkpoint_epoch": 3,
  "raw_dev_macro_f1": 0.7452601739548848,
  "dev_macro_f1": 0.774542334507111,
  "threshold": 0.2,
  "calibration": {
    "threshold": 0.2,
    "calibration_selected": true,
    "grid": [
      0.2,
      0.25,
      0.3,
      0.35,
      0.4,
      0.45,
      0.5,
      0.55,
      0.6,
      0.65,
      0.7,
      0.75,
      0.8
    ],
    "raw_dev_macro_f1": 0.7452601739548848,
    "selected_dev_macro_f1": 0.774542334507111,
    "best_fitted_threshold": 0.2,
    "best_fitted_dev_macro_f1": 0.774542334507111,
   

/kaggle/working/cs781_phase4/P4_01_A_Full_FAST_V2/submission_P4_01_A_Full_FAST_V2.zip

/kaggle/working/cs781_phase4/P4_01_A_Full_FAST_V2/observations.zip

Preserve the COMPLETE saved output for recovery/KD. An observations ZIP has no model weights.


[Markdown Cell]

After a verified completion, submit the printed submission_*.zip to **Phase 4** Codabench. Save the complete Kaggle output, including best/slot .pt files and .json receipts, code, parent_source, tokenizer, job/config, predictions and reports. best.pt is the dev-selected task model; slot_*.pt are exact latest recovery states. Dev F1 is measured locally; private Codabench F1 remains unknown until submission.

A timed stop is safe but incomplete. Resume it before claiming this required method is completed. A failed export can be retried with MODE='export' after attaching the same complete task output; its data/model identities remain checked.